In [1]:
import pandas as pd
import numpy as np

# File path
INVENTORY_FILE = "../WVS_analysis_output/WVS_variable_inventory.csv"

# Load variable inventory
inventory = pd.read_csv(INVENTORY_FILE)

print("Total number of variables:", len(inventory))
print("\nVariable categories:")
display(inventory["Preliminary category"].value_counts())

display(inventory.head())

Total number of variables: 613

Variable categories:


Preliminary category
Survey question / question-derived field           372
Other / requires codebook review                   184
External country-level indicators                   32
Survey metadata / sampling / respondent context     17
Possible derived index / scale                       8
Name: count, dtype: int64

,Column number,Variable,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category
0,1,version,str,0,0.0,1,0,NaN,6-0-0 (2024-04-30),Other / requires codebook review
1,2,doi,str,0,0.0,1,0,NaN,doi.org/10.14281/18241.24,Other / requires codebook review
2,3,A_WAVE,int64,0,0.0,1,0,NaN,7,Survey metadata / sampling / respondent context
3,4,A_YEAR,int64,0,0.0,7,0,NaN,2018 | 2017 | 2021 | 2020 | 2019 | 2022 | 2023,Survey metadata / sampling / respondent context
4,5,A_STUDY,int64,0,0.0,1,0,NaN,2,Survey metadata / sampling / respondent context


In [2]:
# Create an initial review status for all variables

inventory["Review status"] = "Needs review"

# Target variable
inventory.loc[
    inventory["Variable"] == "Q49",
    "Review status"
] = "Target variable"

# Variables that should not be used as ordinary predictors
# These are examples to review, not an automatic final exclusion list.

known_non_predictor_candidates = [
    "Q49"
]

print("Review status:")
display(inventory["Review status"].value_counts())

Review status:


Review status
Needs review       612
Target variable      1
Name: count, dtype: int64

In [3]:
# Count variables in each preliminary category
category_summary = (
    inventory["Preliminary category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Number of variables")
)

category_summary["Percentage"] = (
    category_summary["Number of variables"]
    / len(inventory) * 100
).round(2)

display(category_summary)

,Category,Number of variables,Percentage
0,Survey question / question-derived field,372,60.69
1,Other / requires codebook review,184,30.02
2,External country-level indicators,32,5.22
3,Survey metadata / sampling / respondent context,17,2.77
4,Possible derived index / scale,8,1.31


In [5]:
import pandas as pd

DATA_FILE = "../data/WVS_Cross-National_Wave_7_csv_v6_0.csv"

year_country = pd.read_csv(
    DATA_FILE,
    usecols=["B_COUNTRY_ALPHA", "A_YEAR"]
)

year_summary = (
    year_country
    .groupby(["A_YEAR", "B_COUNTRY_ALPHA"])
    .size()
    .reset_index(name="Respondents")
)

display(year_summary.sort_values(["A_YEAR", "B_COUNTRY_ALPHA"]))

print("Number of survey years:", year_country["A_YEAR"].nunique())
print("Number of countries/regions:", year_country["B_COUNTRY_ALPHA"].nunique())

,A_YEAR,B_COUNTRY_ALPHA,Respondents
0,2017,ARG,1003
1,2017,BOL,2067
2,2017,GRC,1200
3,2017,RUS,1810
4,2017,SRB,1046
...,...,...,...
61,2022,NLD,2145
62,2022,SVK,1200
63,2022,URY,1000
64,2022,UZB,1250


Number of survey years: 7
Number of countries/regions: 66


In [7]:
import pandas as pd
import numpy as np

INVENTORY_FILE = "../WVS_analysis_output/WVS_variable_inventory.csv"

inventory = pd.read_csv(INVENTORY_FILE)

survey_variables = inventory[
    inventory["Preliminary category"]
    == "Survey question / question-derived field"
].copy()

print("Total variables:", len(inventory))
print("Survey-related variables:", len(survey_variables))

display(survey_variables.head())

# Get the list of survey-related variables
survey_variable_names = survey_variables["Variable"].tolist()

# Candidate special codes
special_codes = [-1, -2, -3, -4, -5]

print("Number of survey variables:", len(survey_variable_names))
print("Special codes to inspect:", special_codes)

Total variables: 613
Survey-related variables: 372


,Column number,Variable,Data type,Missing count (NaN),Missing percentage (NaN),Unique values (excluding NaN),Candidate special-code count,Candidate special-code details,Example values,Preliminary category
40,41,Q1,int64,0,0.0,7,155,-1: 27; -2: 61; -5: 67,1 | 2 | 3 | 4 | -2 | -1 | -5,Survey question / question-derived field
41,42,Q2,int64,0,0.0,7,312,-1: 84; -2: 141; -5: 87,1 | 2 | 3 | 4 | -1 | -2 | -5,Survey question / question-derived field
42,43,Q3,int64,0,0.0,7,535,-1: 202; -2: 196; -5: 137,1 | 2 | 3 | 4 | -1 | -2 | -5,Survey question / question-derived field
43,44,Q4,int64,0,0.0,7,1090,-1: 587; -2: 323; -5: 180,3 | 4 | 2 | 1 | -2 | -1 | -5,Survey question / question-derived field
44,45,Q5,int64,0,0.0,7,1160,-1: 368; -2: 289; -5: 503,1 | 3 | 2 | 4 | -2 | -1 | -5,Survey question / question-derived field


Number of survey variables: 372
Special codes to inspect: [-1, -2, -3, -4, -5]


In [8]:
survey_variable_names = survey_variables["Variable"].tolist()

special_codes = [-1, -2, -3, -4, -5]

print("Number of survey variables:", len(survey_variable_names))
print("Special codes:", special_codes)

Number of survey variables: 372
Special codes: [-1, -2, -3, -4, -5]


In [9]:
DATA_FILE = "../data/WVS_Cross-National_Wave_7_csv_v6_0.csv"

survey_data = pd.read_csv(
    DATA_FILE,
    usecols=survey_variable_names
)

print("Survey data shape:", survey_data.shape)

Survey data shape: (97220, 372)


In [10]:
special_code_summary = pd.DataFrame(
    index=survey_variable_names
)

for code in special_codes:
    special_code_summary[f"Code_{code}"] = (
        survey_data == code
    ).sum()

special_code_summary["NaN_count"] = (
    survey_data.isna().sum()
)

special_code_summary["Special_code_count"] = (
    special_code_summary[
        [f"Code_{code}" for code in special_codes]
    ].sum(axis=1)
)

special_code_summary["Total_missing_count"] = (
    special_code_summary["Special_code_count"]
    + special_code_summary["NaN_count"]
)

special_code_summary["Missing_percentage"] = (
    special_code_summary["Total_missing_count"]
    / len(survey_data) * 100
).round(2)

special_code_summary = special_code_summary.sort_values(
    "Missing_percentage",
    ascending=False
)

display(special_code_summary.head(30))

,Code_-1,Code_-2,Code_-3,Code_-4,Code_-5,NaN_count,Special_code_count,Total_missing_count,Missing_percentage
Q82_ECO,411,15,0,95721,0,0,96147,96147,98.90
Q82_OAS,171,23,0,95820,0,0,96014,96014,98.76
Q82_UNDP,1244,55,0,93620,0,0,94919,94919,97.63
Q82_SAARC,578,35,0,94186,0,0,94799,94799,97.51
Q82_TLC,262,8,0,94479,0,0,94749,94749,97.46
Q82_NAFTA,11,52,0,94624,0,0,94687,94687,97.39
Q82_APEC,526,9,0,93309,0,0,93844,93844,96.53
Q82_CUSMA,0,0,0,93202,0,0,93202,93202,95.87
Q82_AFRICANUNION,593,25,0,92272,10,0,92900,92900,95.56
Q82_GULFCOOP,1679,105,0,90009,0,0,91793,91793,94.42


In [11]:
country_codes = pd.read_csv(
    DATA_FILE,
    usecols=["B_COUNTRY_ALPHA"]
)

print("Number of countries:", 
      country_codes["B_COUNTRY_ALPHA"].nunique())

Number of countries: 66


In [12]:
variables_to_check = [
    "Q82_ECO",
    "Q82_OAS",
    "Q82_UNDP",
    "Q294A",
    "Q294B",
    "Q292I",
    "Q292J"
]

country_variable_data = pd.read_csv(
    DATA_FILE,
    usecols=["B_COUNTRY_ALPHA"] + variables_to_check
)

In [13]:
country_coverage = []

for variable in variables_to_check:

    valid = (
        country_variable_data[variable]
        .notna()
        & ~country_variable_data[variable].isin(special_codes)
    )

    temp = country_variable_data.loc[
        valid, ["B_COUNTRY_ALPHA"]
    ]

    counts = temp["B_COUNTRY_ALPHA"].value_counts()

    country_coverage.append({
        "Variable": variable,
        "Countries with valid responses": len(counts),
        "Respondents with valid responses": int(counts.sum()),
        "Countries": ", ".join(counts.index.tolist())
    })

country_coverage_df = pd.DataFrame(country_coverage)

display(country_coverage_df)

,Variable,Countries with valid responses,Respondents with valid responses,Countries
0,Q82_ECO,1,1073,IRN
1,Q82_OAS,1,1206,PER
2,Q82_UNDP,3,2301,"LBN, IRQ, EGY"
3,Q294A,12,7182,"IRN, ZWE, MMR, PHL, VEN, VNM, KGZ, NIC, KEN, L..."
4,Q294B,12,7224,"IRN, NIC, KGZ, MAR, MMR, PHL, VNM, VEN, KEN, L..."
5,Q292I,13,15369,"IRN, KEN, MMR, TJK, PHL, VEN, MAR, ZWE, ETH, L..."
6,Q292J,13,15421,"IRN, KEN, MAR, MMR, PHL, TJK, ZWE, VEN, ETH, L..."


The following 55 candidate variables are grouped by their broad themes. They were selected to explore a range of potential factors associated with life satisfaction, including personal values, social attitudes, family norms, and material wellbeing. The selection is based on the WVS Wave 7 Codebook.

### Candidate Variable Groups

| Category | Variables | Main content | Reason for inclusion |
|---|---|---|---|
| **Personal Values** | Q1–Q6 | Importance of family, friends, leisure, politics, work, and religion. | To explore whether people's priorities and values are associated with life satisfaction. |
| **Child-rearing Values** | Q7–Q17 | Qualities considered important for children, such as independence, hard work, responsibility, tolerance, and obedience. | To investigate whether broader social and personal values are related to life satisfaction. |
| **Social Tolerance** | Q18–Q26 | Attitudes towards different groups as potential neighbours, including immigrants, people of different religions, and other social groups. | To examine whether social openness and tolerance are associated with life satisfaction. |
| **Family and Gender Norms** | Q27–Q38 | Views on family responsibilities, gender roles, work, parenting, and intergenerational care. | To explore how family expectations and gender-related attitudes relate to life satisfaction. |
| **Work and Social Attitudes** | Q39–Q45 | Attitudes towards work, social change, technology, and authority. | To investigate whether work-related beliefs and attitudes towards social change provide additional explanatory information. |
| **Wellbeing** | Q46–Q48 | Happiness, self-reported health, and perceived freedom of choice and control over life. | To examine personal wellbeing and perceived autonomy as potential predictors. |
| **Economic Security** | Q50–Q56 | Satisfaction with household finances, experiences of material deprivation, personal security, and living standards compared with parents. | To explore whether financial satisfaction and material security provide information beyond household income. |

### Selection Rationale

These variables were selected to go beyond traditional economic and demographic indicators. They cover several dimensions of life that may be associated with life satisfaction:

- **Personal and social dimensions:** values, relationships, tolerance, and family expectations.
- **Psychological and individual dimensions:** happiness, health, and perceived control over life.
- **Economic and material dimensions:** financial satisfaction, deprivation, and living standards.

The purpose is to explore both expected and less obvious associations with life satisfaction, rather than assuming in advance which factors matter most.

**Note:** Q49 (Life Satisfaction) is excluded because it is the target variable. Q46 (Happiness) is retained as a potential candidate for a separate comparison, but excluded from the primary feature set because it is conceptually very close to life satisfaction. All candidates will be assessed for missingness, country coverage, and measurement type before final feature selection.

In [14]:
candidate_groups = {
    "Personal Values": [
        "Q1", "Q2", "Q3", "Q4", "Q5", "Q6"
    ],

    "Child-rearing Values": [
        "Q7", "Q8", "Q9", "Q10", "Q11",
        "Q12", "Q13", "Q14", "Q15", "Q16", "Q17"
    ],

    "Social Tolerance": [
        "Q18", "Q19", "Q20", "Q21", "Q22",
        "Q23", "Q24", "Q25", "Q26"
    ],

    "Family and Gender Norms": [
        "Q27", "Q28", "Q29", "Q30", "Q31",
        "Q32", "Q33", "Q34", "Q35", "Q36",
        "Q37", "Q38"
    ],

    "Work and Social Attitudes": [
        "Q39", "Q40", "Q41", "Q42", "Q43", "Q44", "Q45"
    ],

    "Wellbeing": [
        "Q46", "Q47", "Q48"
    ],

    "Economic Security": [
        "Q50", "Q51", "Q52", "Q53", "Q54", "Q55", "Q56"
    ]
}

candidate_variables = [
    variable
    for variables_in_group in candidate_groups.values()
    for variable in variables_in_group
]

print("Candidate variables:", len(candidate_variables))
print("Unique candidate variables:", len(set(candidate_variables)))

Candidate variables: 55
Unique candidate variables: 55


In [17]:
# Create a table containing all candidate variables
candidate_variables = [
    variable
    for group in candidate_groups.values()
    for variable in group
]

# Extract candidate variables from the inventory
candidate_feature_table = inventory[
    inventory["Variable"].isin(candidate_variables)
].copy()

# Add special-code information
special_summary = (
    special_code_summary
    .rename_axis("Variable")
    .reset_index()
)

candidate_feature_table = candidate_feature_table.merge(
    special_summary,
    on="Variable",
    how="left"
)

# Check whether all candidate variables were found
missing_from_inventory = set(candidate_variables) - set(
    candidate_feature_table["Variable"]
)

print("Number of candidate variables:", len(candidate_variables))
print("Variables found in inventory:", len(candidate_feature_table))
print("Variables missing from inventory:", missing_from_inventory)

# Display the most relevant columns
columns_to_display = [
    "Variable",
    "Data type",
    "Missing percentage (NaN)",
    "Missing_percentage",
    "Special_code_count",
    "Total_missing_count"
]

display(
    candidate_feature_table[columns_to_display]
    .sort_values("Missing_percentage", ascending=False)
)

Number of candidate variables: 55
Variables found in inventory: 55
Variables missing from inventory: set()


,Variable,Data type,Missing percentage (NaN),Missing_percentage,Special_code_count,Total_missing_count
35,Q36,int64,0.0,16.93,16459,16459
21,Q22,int64,0.0,4.26,4137,4137
41,Q42,int64,0.0,3.41,3318,3318
31,Q32,int64,0.0,3.29,3201,3201
44,Q45,int64,0.0,2.80,2724,2724
28,Q29,int64,0.0,2.72,2641,2641
24,Q25,int64,0.0,2.51,2443,2443
42,Q43,int64,0.0,2.20,2137,2137
30,Q31,int64,0.0,2.13,2068,2068
43,Q44,int64,0.0,2.12,2063,2063


In [18]:
# Load country identifiers and candidate variables
columns_to_load = ["B_COUNTRY_ALPHA"] + candidate_variables

country_variable_data = pd.read_csv(
    DATA_FILE,
    usecols=columns_to_load
)

# Identify valid responses
valid_data = country_variable_data.copy()

valid_data[candidate_variables] = valid_data[
    candidate_variables
].where(
    ~valid_data[candidate_variables].isin(special_codes)
)

# Calculate country coverage for each variable
coverage_results = []

total_countries = country_variable_data["B_COUNTRY_ALPHA"].nunique()

for variable in candidate_variables:

    valid_countries = valid_data.loc[
        valid_data[variable].notna(),
        "B_COUNTRY_ALPHA"
    ].nunique()

    valid_respondents = valid_data[variable].notna().sum()

    coverage_results.append({
        "Variable": variable,
        "Countries_with_valid_responses": valid_countries,
        "Country_coverage_percentage": round(
            valid_countries / total_countries * 100, 2
        ),
        "Respondents_with_valid_responses": valid_respondents
    })

coverage_table = pd.DataFrame(coverage_results)

coverage_table = coverage_table.sort_values(
    "Country_coverage_percentage"
)

print("Total countries:", total_countries)

display(coverage_table)

Total countries: 66


,Variable,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses
35,Q36,57,86.36,80761
21,Q22,64,96.97,93083
24,Q25,65,98.48,94777
0,Q1,66,100.00,97065
4,Q5,66,100.00,96060
5,Q6,66,100.00,96294
2,Q3,66,100.00,96685
1,Q2,66,100.00,96908
8,Q9,66,100.00,96528
9,Q10,66,100.00,96667


In [20]:
# Combine missingness and country coverage information
candidate_screening_table = candidate_feature_table.merge(
    coverage_table,
    on="Variable",
    how="left"
)

# Select relevant columns
screening_columns = [
    "Variable",
    "Missing_percentage",
    "Countries_with_valid_responses",
    "Country_coverage_percentage",
    "Respondents_with_valid_responses"
]

candidate_screening_table = candidate_screening_table[
    screening_columns
].sort_values(
    ["Country_coverage_percentage", "Missing_percentage"],
    ascending=[True, False]
)

candidate_screening_table = candidate_screening_table.reset_index(drop=True)

display(candidate_screening_table)

,Variable,Missing_percentage,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses
0,Q36,16.93,57,86.36,80761
1,Q22,4.26,64,96.97,93083
2,Q25,2.51,65,98.48,94777
3,Q42,3.41,66,100.00,93902
4,Q32,3.29,66,100.00,94019
5,Q45,2.80,66,100.00,94496
6,Q29,2.72,66,100.00,94579
7,Q43,2.20,66,100.00,95083
8,Q31,2.13,66,100.00,95152
9,Q44,2.12,66,100.00,95157


### Variable Measurement Type Analysis

In [21]:
# Define measurement types based on the WVS Codebook

measurement_groups = {
    "Ordinal": [
        "Q1", "Q2", "Q3", "Q4", "Q5", "Q6",
        "Q27", "Q28", "Q29", "Q30", "Q31", "Q32",
        "Q33", "Q34", "Q35", "Q36", "Q37", "Q38",
        "Q39", "Q40", "Q41",
        "Q43", "Q44", "Q45",
        "Q46", "Q47", "Q48",
        "Q50", "Q51", "Q52", "Q53", "Q54", "Q55", "Q56"
    ],

    "Binary": [
        "Q7", "Q8", "Q9", "Q10", "Q11", "Q12",
        "Q13", "Q14", "Q15", "Q16", "Q17",
        "Q18", "Q19", "Q20", "Q21", "Q22",
        "Q23", "Q24", "Q25", "Q26"
    ],

    "Nominal": [
        "Q42"
    ]
}

# Create a variable-to-measurement-type mapping
measurement_mapping = {
    variable: measurement_type
    for measurement_type, variables in measurement_groups.items()
    for variable in variables
}

# Add measurement type to the screening table
candidate_screening_table["Measurement_type"] = (
    candidate_screening_table["Variable"].map(measurement_mapping)
)

# Check the classification
print("Total candidate variables:", len(candidate_variables))
print(
    "Variables classified:",
    candidate_screening_table["Measurement_type"].notna().sum()
)
print(
    "Unclassified variables:",
    candidate_screening_table.loc[
        candidate_screening_table["Measurement_type"].isna(),
        "Variable"
    ].tolist()
)

display(
    candidate_screening_table[
        ["Variable", "Measurement_type"]
    ].sort_values("Variable")
)

Total candidate variables: 55
Variables classified: 55
Unclassified variables: []


,Variable,Measurement_type
54,Q1,Ordinal
48,Q10,Binary
18,Q11,Binary
43,Q12,Binary
26,Q13,Binary
24,Q14,Binary
27,Q15,Binary
25,Q16,Binary
19,Q17,Binary
32,Q18,Binary


In [22]:
# Inspect observed values for each candidate variable

value_inspection = []

for variable in candidate_variables:

    values = sorted(
        survey_data[variable]
        .dropna()
        .unique()
        .tolist()
    )

    value_inspection.append({
        "Variable": variable,
        "Measurement_type": measurement_mapping.get(variable),
        "Unique_value_count": len(values),
        "Observed_values": values
    })

value_inspection_table = pd.DataFrame(value_inspection)

display(
    value_inspection_table.sort_values("Variable")
)

,Variable,Measurement_type,Unique_value_count,Observed_values
0,Q1,Ordinal,7,"[-5, -2, -1, 1, 2, 3, 4]"
9,Q10,Binary,5,"[-5, -2, -1, 1, 2]"
10,Q11,Binary,5,"[-5, -2, -1, 1, 2]"
11,Q12,Binary,5,"[-5, -2, -1, 1, 2]"
12,Q13,Binary,5,"[-5, -2, -1, 1, 2]"
13,Q14,Binary,5,"[-5, -2, -1, 1, 2]"
14,Q15,Binary,5,"[-5, -2, -1, 1, 2]"
15,Q16,Binary,5,"[-5, -2, -1, 1, 2]"
16,Q17,Binary,5,"[-5, -2, -1, 1, 2]"
17,Q18,Binary,5,"[-5, -2, -1, 1, 2]"


In [23]:
# Create a final preliminary measurement dictionary

measurement_dictionary = candidate_screening_table.merge(
    value_inspection_table[
        [
            "Variable",
            "Unique_value_count",
            "Observed_values"
        ]
    ],
    on="Variable",
    how="left"
)

# Display the dictionary
measurement_dictionary = measurement_dictionary.sort_values(
    "Variable"
).reset_index(drop=True)

display(measurement_dictionary)

,Variable,Missing_percentage,Countries_with_valid_responses,Country_coverage_percentage,Respondents_with_valid_responses,Measurement_type,Unique_value_count,Observed_values
0,Q1,0.16,66,100.00,97065,Ordinal,7,"[-5, -2, -1, 1, 2, 3, 4]"
1,Q10,0.57,66,100.00,96667,Binary,5,"[-5, -2, -1, 1, 2]"
2,Q11,1.24,66,100.00,96017,Binary,5,"[-5, -2, -1, 1, 2]"
3,Q12,0.66,66,100.00,96577,Binary,5,"[-5, -2, -1, 1, 2]"
4,Q13,1.15,66,100.00,96104,Binary,5,"[-5, -2, -1, 1, 2]"
5,Q14,1.16,66,100.00,96089,Binary,5,"[-5, -2, -1, 1, 2]"
6,Q15,1.14,66,100.00,96109,Binary,5,"[-5, -2, -1, 1, 2]"
7,Q16,1.16,66,100.00,96096,Binary,5,"[-5, -2, -1, 1, 2]"
8,Q17,1.23,66,100.00,96028,Binary,5,"[-5, -2, -1, 1, 2]"
9,Q18,0.96,66,100.00,96291,Binary,5,"[-5, -2, -1, 1, 2]"


In [24]:
measurement_dictionary.to_csv(
    "../WVS_analysis_output/WVS_candidate_measurement_dictionary.csv",
    index=False
)

### Identify remaining survey variables

In [25]:
# Identify remaining survey variables
remaining_variables = [
    variable
    for variable in survey_variable_names
    if variable not in candidate_variables
]

print("Total survey-related variables:", len(survey_variable_names))
print("Already inspected:", len(candidate_variables))
print("Remaining variables:", len(remaining_variables))

Total survey-related variables: 372
Already inspected: 55
Remaining variables: 317


In [26]:
# Explore the preliminary categories of remaining variables

remaining_inventory = inventory[
    inventory["Variable"].isin(remaining_variables)
].copy()

display(
    remaining_inventory[
        ["Variable", "Data type", "Preliminary category"]
    ].head(50)
)

print(
    remaining_inventory["Preliminary category"]
    .value_counts()
)

,Variable,Data type,Preliminary category
73,Q33_3,int64,Survey question / question-derived field
75,Q34_3,int64,Survey question / question-derived field
77,Q35_3,int64,Survey question / question-derived field
91,Q49,int64,Survey question / question-derived field
99,Q57,int64,Survey question / question-derived field
100,Q58,int64,Survey question / question-derived field
101,Q59,int64,Survey question / question-derived field
102,Q60,int64,Survey question / question-derived field
103,Q61,int64,Survey question / question-derived field
104,Q62,int64,Survey question / question-derived field


Preliminary category
Survey question / question-derived field    317
Name: count, dtype: int64


In [27]:
# Classify remaining variables by naming patterns

def classify_variable_name(variable):

    if variable == "Q49":
        return "Target variable"

    elif any(variable.startswith(q) for q in [
        "Q82_", "Q292_", "Q294_"
    ]):
        return "Country-specific / question variant"

    elif "_" in variable:
        return "Alternative / derived question field"

    else:
        return "Standard survey question"


remaining_inventory["Variable_pattern"] = (
    remaining_inventory["Variable"].apply(
        classify_variable_name
    )
)

print(
    remaining_inventory["Variable_pattern"]
    .value_counts()
)

display(
    remaining_inventory[
        ["Variable", "Variable_pattern"]
    ].head(100)
)

Variable_pattern
Standard survey question                294
Country-specific / question variant      17
Alternative / derived question field      5
Target variable                           1
Name: count, dtype: int64


,Variable,Variable_pattern
73,Q33_3,Alternative / derived question field
75,Q34_3,Alternative / derived question field
77,Q35_3,Alternative / derived question field
91,Q49,Target variable
99,Q57,Standard survey question
...,...,...
190,Q119,Standard survey question
191,Q120,Standard survey question
192,Q121,Standard survey question
193,Q122,Standard survey question


In [28]:
# Load remaining survey variables and country identifiers

columns_to_load = (
    remaining_variables + ["B_COUNTRY_ALPHA"]
)

remaining_data = pd.read_csv(
    DATA_FILE,
    usecols=columns_to_load
)

print("Dataset shape:", remaining_data.shape)
print("Number of respondents:", len(remaining_data))

Dataset shape: (97220, 318)
Number of respondents: 97220


In [31]:
# Calculate preliminary missingness statistics

special_codes = [-1, -2, -3, -4, -5]

remaining_quality = pd.DataFrame(
    index=remaining_variables
)

for code in special_codes:
    remaining_quality[f"Code_{code}"] = (
        remaining_data[remaining_variables] == code
    ).sum()

remaining_quality["NaN_count"] = (
    remaining_data[remaining_variables].isna().sum()
)

remaining_quality["Special_code_count"] = (
    remaining_quality[
        [f"Code_{code}" for code in special_codes]
    ].sum(axis=1)
)

remaining_quality["Total_missing_count"] = (
    remaining_quality["Special_code_count"]
    + remaining_quality["NaN_count"]
)

remaining_quality["Missing_percentage"] = (
    remaining_quality["Total_missing_count"]
    / len(remaining_data) * 100
).round(2)

remaining_quality = (
    remaining_quality
    .sort_values("Missing_percentage", ascending=False)
)

display(remaining_quality.head(30))

,Code_-1,Code_-2,Code_-3,Code_-4,Code_-5,NaN_count,Special_code_count,Total_missing_count,Missing_percentage
Q82_ECO,411,15,0,95721,0,0,96147,96147,98.90
Q82_OAS,171,23,0,95820,0,0,96014,96014,98.76
Q82_UNDP,1244,55,0,93620,0,0,94919,94919,97.63
Q82_SAARC,578,35,0,94186,0,0,94799,94799,97.51
Q82_TLC,262,8,0,94479,0,0,94749,94749,97.46
Q82_NAFTA,11,52,0,94624,0,0,94687,94687,97.39
Q82_APEC,526,9,0,93309,0,0,93844,93844,96.53
Q82_CUSMA,0,0,0,93202,0,0,93202,93202,95.87
Q82_AFRICANUNION,593,25,0,92272,10,0,92900,92900,95.56
Q82_GULFCOOP,1679,105,0,90009,0,0,91793,91793,94.42


In [30]:
# Identify valid responses across countries

valid_remaining_data = remaining_data.copy()

valid_remaining_data[remaining_variables] = (
    valid_remaining_data[remaining_variables].replace(
        special_codes, np.nan
    )
)

total_countries = (
    remaining_data["B_COUNTRY_ALPHA"].nunique()
)

coverage_results = []

for variable in remaining_variables:

    valid_mask = valid_remaining_data[variable].notna()

    valid_countries = (
        valid_remaining_data.loc[
            valid_mask, "B_COUNTRY_ALPHA"
        ].nunique()
    )

    coverage_results.append({
        "Variable": variable,
        "Valid_respondents": int(valid_mask.sum()),
        "Countries_with_valid_responses": valid_countries,
        "Country_coverage_percentage": round(
            valid_countries / total_countries * 100, 2
        )
    })

remaining_coverage = pd.DataFrame(coverage_results)

display(
    remaining_coverage.sort_values(
        "Country_coverage_percentage"
    ).head(30)
)

,Variable,Valid_respondents,Countries_with_valid_responses,Country_coverage_percentage
35,Q82_CUSMA,4018,1,1.52
36,Q82_ECO,1073,1,1.52
42,Q82_OAS,1206,1,1.52
41,Q82_NAFTA,2533,1,1.52
45,Q82_TLC,2471,2,3.03
43,Q82_SAARC,2421,2,3.03
31,Q82_APEC,3376,3,4.55
46,Q82_UNDP,2301,3,4.55
30,Q82_AFRICANUNION,4320,4,6.06
44,Q82_SCO,5773,5,7.58


In [32]:
print(remaining_quality.index[:5])
print(remaining_quality.columns.tolist())
print(remaining_quality["Special_code_count"].head())

Index(['Q82_ECO', 'Q82_OAS', 'Q82_UNDP', 'Q82_SAARC', 'Q82_TLC'], dtype='str')
['Code_-1', 'Code_-2', 'Code_-3', 'Code_-4', 'Code_-5', 'NaN_count', 'Special_code_count', 'Total_missing_count', 'Missing_percentage']
Q82_ECO      96147
Q82_OAS      96014
Q82_UNDP     94919
Q82_SAARC    94799
Q82_TLC      94749
Name: Special_code_count, dtype: int64


In [33]:
# Extract variables belonging to Q57-Q105

import re

def get_question_number(variable):
    match = re.match(r"Q(\d+)", variable)
    return int(match.group(1)) if match else None

remaining_inventory["Question_number"] = (
    remaining_inventory["Variable"].apply(
        get_question_number
    )
)

q57_q105 = remaining_inventory[
    remaining_inventory["Question_number"].between(57, 105)
].copy()

print("Number of variables:", len(q57_q105))

display(
    q57_q105[
        ["Variable", "Data type", "Variable_pattern"]
    ].reset_index(drop=True)
)

Number of variables: 78


,Variable,Data type,Variable_pattern
0,Q57,int64,Standard survey question
1,Q58,int64,Standard survey question
2,Q59,int64,Standard survey question
3,Q60,int64,Standard survey question
4,Q61,int64,Standard survey question
...,...,...,...
73,Q103R,int64,Standard survey question
74,Q104,int64,Standard survey question
75,Q104R,int64,Standard survey question
76,Q105,int64,Standard survey question


In [34]:
# Combine quality and coverage information for Q57-Q105

q57_q105_quality = (
    q57_q105[["Variable"]]
    .merge(
        remaining_quality.rename_axis("Variable").reset_index(),
        on="Variable",
        how="left"
    )
    .merge(
        remaining_coverage,
        on="Variable",
        how="left"
    )
)

display(
    q57_q105_quality[
        [
            "Variable",
            "Missing_percentage",
            "Countries_with_valid_responses",
            "Country_coverage_percentage",
            "Valid_respondents"
        ]
    ].sort_values(
        "Country_coverage_percentage",
        ascending=True
    ).reset_index(drop=True)
)

,Variable,Missing_percentage,Countries_with_valid_responses,Country_coverage_percentage,Valid_respondents
0,Q82_CUSMA,95.87,1,1.52,4018
1,Q82_OAS,98.76,1,1.52,1206
2,Q82_NAFTA,97.39,1,1.52,2533
3,Q82_ECO,98.90,1,1.52,1073
4,Q82_SAARC,97.51,2,3.03,2421
...,...,...,...,...,...
73,Q103R,1.65,66,100.00,95618
74,Q104,3.13,66,100.00,94179
75,Q104R,3.13,66,100.00,94179
76,Q105,6.57,66,100.00,90828


In [35]:
# Separate standard and country-specific variables

q57_q105_standard = q57_q105[
    q57_q105["Variable_pattern"]
    == "Standard survey question"
].copy()

q57_q105_standard_quality = (
    q57_q105_standard[["Variable"]]
    .merge(
        remaining_quality.rename_axis("Variable").reset_index(),
        on="Variable",
        how="left"
    )
    .merge(
        remaining_coverage,
        on="Variable",
        how="left"
    )
)

print(
    "Standard survey variables:",
    len(q57_q105_standard_quality)
)

print(
    "Variables covering all countries:",
    (
        q57_q105_standard_quality[
            "Countries_with_valid_responses"
        ] == 66
    ).sum()
)

display(
    q57_q105_standard_quality[
        [
            "Variable",
            "Missing_percentage",
            "Countries_with_valid_responses",
            "Country_coverage_percentage",
            "Valid_respondents"
        ]
    ].sort_values(
        "Missing_percentage",
        ascending=False
    )
)

Standard survey variables: 61
Variables covering all countries: 44


,Variable,Missing_percentage,Countries_with_valid_responses,Country_coverage_percentage,Valid_respondents
36,Q93,37.33,61,92.42,60926
35,Q92,36.79,64,96.97,61450
34,Q91,36.25,64,96.97,61974
29,Q86,25.90,62,93.94,72041
25,Q82,22.15,60,90.91,75682
...,...,...,...,...,...
38,Q94R,1.13,66,100.00,96126
37,Q94,1.13,66,100.00,96126
2,Q59,0.85,66,100.00,96393
3,Q60,0.60,66,100.00,96635


## 4. Feature Discovery: Social Capital, Trust and Organizational Membership

This section examines WVS variables Q57–Q105.

The objective is to identify potentially relevant social, trust-related,
and organizational factors associated with life satisfaction.

Variables will be evaluated based on:
- Conceptual relevance
- Missingness
- Country coverage
- Measurement type
- Potential overlap with other variables

In [38]:
# Merge variable inventory with quality information
q57_q105_standard_quality = (
    q57_q105_standard
    .merge(
        q57_q105_quality,
        on="Variable",
        how="left"
    )
)

print("Number of variables:", len(q57_q105_standard_quality))

Number of variables: 61


In [39]:
display(
    q57_q105_standard_quality[
        [
            "Variable",
            "Missing_percentage",
            "Countries_with_valid_responses",
            "Country_coverage_percentage",
            "Valid_respondents"
        ]
    ].sort_values(
        by="Missing_percentage",
        ascending=False
    )
)

,Variable,Missing_percentage,Countries_with_valid_responses,Country_coverage_percentage,Valid_respondents
36,Q93,37.33,61,92.42,60926
35,Q92,36.79,64,96.97,61450
34,Q91,36.25,64,96.97,61974
29,Q86,25.90,62,93.94,72041
25,Q82,22.15,60,90.91,75682
...,...,...,...,...,...
38,Q94R,1.13,66,100.00,96126
37,Q94,1.13,66,100.00,96126
2,Q59,0.85,66,100.00,96393
3,Q60,0.60,66,100.00,96635


In [40]:
# Sort all 61 variables by missing percentage
q57_q105_sorted = q57_q105_standard_quality.sort_values(
    by="Missing_percentage",
    ascending=False
)

# Print all rows without truncation
print(
    q57_q105_sorted[
        [
            "Variable",
            "Missing_percentage",
            "Countries_with_valid_responses",
            "Country_coverage_percentage",
            "Valid_respondents"
        ]
    ].to_string(index=False)
)

Variable  Missing_percentage  Countries_with_valid_responses  Country_coverage_percentage  Valid_respondents
     Q93               37.33                              61                        92.42              60926
     Q92               36.79                              64                        96.97              61450
     Q91               36.25                              64                        96.97              61974
     Q86               25.90                              62                        93.94              72041
     Q82               22.15                              60                        90.91              75682
     Q89               21.85                              64                        96.97              75977
     Q85               20.18                              65                        98.48              77597
     Q87               19.77                              64                        96.97              77995
     Q84           

## 5. Conceptual Classification of Q57-Q105

Variables are classified into the following conceptual groups:

1. Generalized and interpersonal trust
2. Institutional confidence
3. International organizational attitudes and knowledge
4. Social and organizational membership
5. Alternative recoded variables

The objective is to identify conceptually distinct predictors
and avoid including redundant versions of the same variables.

In [41]:
# Create conceptual variable groups

trust_variables = [
    "Q57",
    "Q58", "Q59", "Q60", "Q61", "Q62", "Q63"
]

institutional_confidence = [
    f"Q{i}" for i in range(64, 90)
]

international_attitudes_knowledge = [
    "Q90", "Q91", "Q92", "Q93"
]

organization_membership = [
    f"Q{i}" for i in range(94, 106)
]

recoded_membership_variables = [
    f"Q{i}R" for i in range(94, 106)
]

# Print group sizes
print("General and interpersonal trust:", len(trust_variables))
print("Institutional confidence:", len(institutional_confidence))
print("International attitudes and knowledge:",
      len(international_attitudes_knowledge))
print("Organizational membership:", len(organization_membership))
print("Recoded membership variables:",
      len(recoded_membership_variables))

General and interpersonal trust: 7
Institutional confidence: 26
International attitudes and knowledge: 4
Organizational membership: 12
Recoded membership variables: 12


In [42]:
# Create a conceptual classification dictionary

variable_groups = {
    "General and interpersonal trust": trust_variables,
    "Institutional confidence": institutional_confidence,
    "International attitudes and knowledge": international_attitudes_knowledge,
    "Organizational membership": organization_membership,
    "Recoded membership variables": recoded_membership_variables
}

# Convert groups into a DataFrame
classification_rows = []

for group, variables in variable_groups.items():
    for variable in variables:
        classification_rows.append({
            "Variable": variable,
            "Conceptual_group": group
        })

q57_q105_classification = pd.DataFrame(classification_rows)

# Merge with missingness and country coverage information
q57_q105_classification = q57_q105_classification.merge(
    q57_q105_standard_quality[
        [
            "Variable",
            "Missing_percentage",
            "Countries_with_valid_responses",
            "Country_coverage_percentage",
            "Valid_respondents"
        ]
    ],
    on="Variable",
    how="left"
)

display(q57_q105_classification.head(15))

,Variable,Conceptual_group,Missing_percentage,Countries_with_valid_responses,Country_coverage_percentage,Valid_respondents
0,Q57,General and interpersonal trust,1.38,66,100.00,95883
1,Q58,General and interpersonal trust,0.34,66,100.00,96888
2,Q59,General and interpersonal trust,0.85,66,100.00,96393
3,Q60,General and interpersonal trust,0.60,66,100.00,96635
4,Q61,General and interpersonal trust,1.40,66,100.00,95859
5,Q62,General and interpersonal trust,4.19,66,100.00,93144
6,Q63,General and interpersonal trust,4.19,66,100.00,93142
7,Q64,Institutional confidence,2.09,66,100.00,95184
8,Q65,Institutional confidence,4.45,64,96.97,92897
9,Q66,Institutional confidence,2.27,66,100.00,95010


In [43]:
OUTPUT_FILE = "../WVS_analysis_output/WVS_Q57_Q105_conceptual_classification.csv"

q57_q105_classification.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", OUTPUT_FILE)

Saved: ../WVS_analysis_output/WVS_Q57_Q105_conceptual_classification.csv


In [44]:
# 1. General and interpersonal trust
trust_variables = [
    f"Q{i}" for i in range(57, 64)
]

# 2. Institutional confidence
institutional_confidence = (
    [f"Q{i}" for i in range(64, 82)]
    + [f"Q{i}" for i in range(83, 90)]
)

# 3. Country-specific institutional confidence
country_specific_confidence = [
    variable for variable in survey_variable_names
    if variable.startswith("Q82_")
]

# 4. International organization attitudes
international_attitudes = ["Q90"]

# 5. International organization knowledge
international_knowledge = [
    "Q91", "Q92", "Q93"
]

# 6. Original organizational membership variables
organization_membership = [
    f"Q{i}" for i in range(94, 106)
]

# 7. Recoded membership variables
recoded_membership_variables = [
    f"Q{i}R" for i in range(94, 106)
]

# Keep only variables that actually exist in the dataset
variable_groups = {
    "General and interpersonal trust": trust_variables,
    "Institutional confidence": institutional_confidence,
    "Country-specific institutional confidence": country_specific_confidence,
    "International organization attitudes": international_attitudes,
    "International organization knowledge": international_knowledge,
    "Organizational membership": organization_membership,
    "Recoded membership variables": recoded_membership_variables
}

variable_groups = {
    group: [
        variable for variable in variables
        if variable in survey_variable_names
    ]
    for group, variables in variable_groups.items()
}

# Print group sizes and variable names
for group, variables in variable_groups.items():
    print(f"\n{group}: {len(variables)} variables")
    print(variables)


General and interpersonal trust: 7 variables
['Q57', 'Q58', 'Q59', 'Q60', 'Q61', 'Q62', 'Q63']

Institutional confidence: 25 variables
['Q64', 'Q65', 'Q66', 'Q67', 'Q68', 'Q69', 'Q70', 'Q71', 'Q72', 'Q73', 'Q74', 'Q75', 'Q76', 'Q77', 'Q78', 'Q79', 'Q80', 'Q81', 'Q83', 'Q84', 'Q85', 'Q86', 'Q87', 'Q88', 'Q89']

Country-specific institutional confidence: 17 variables
['Q82_AFRICANUNION', 'Q82_APEC', 'Q82_ARABLEAGUE', 'Q82_ASEAN', 'Q82_CIS', 'Q82_CUSMA', 'Q82_ECO', 'Q82_EU', 'Q82_GULFCOOP', 'Q82_ISLCOOP', 'Q82_MERCOSUR', 'Q82_NAFTA', 'Q82_OAS', 'Q82_SAARC', 'Q82_SCO', 'Q82_TLC', 'Q82_UNDP']

International organization attitudes: 1 variables
['Q90']

International organization knowledge: 3 variables
['Q91', 'Q92', 'Q93']

Organizational membership: 12 variables
['Q94', 'Q95', 'Q96', 'Q97', 'Q98', 'Q99', 'Q100', 'Q101', 'Q102', 'Q103', 'Q104', 'Q105']

Recoded membership variables: 12 variables
['Q94R', 'Q95R', 'Q96R', 'Q97R', 'Q98R', 'Q99R', 'Q100R', 'Q101R', 'Q102R', 'Q103R', 'Q104R', 'Q

In [45]:
# Create a variable classification table
classification_rows = []

for group, variables in variable_groups.items():
    for variable in variables:
        classification_rows.append({
            "Variable": variable,
            "Feature group": group
        })

q57_q105_classification = pd.DataFrame(classification_rows)

# Merge missingness and country coverage information
q57_q105_classification = (
    q57_q105_classification
    .merge(
        q57_q105_quality,
        on="Variable",
        how="left"
    )
)

# Display the classification table
q57_q105_classification.to_string(index=False)

'        Variable                             Feature group  Code_-1  Code_-2  Code_-3  Code_-4  Code_-5  NaN_count  Special_code_count  Total_missing_count  Missing_percentage  Valid_respondents  Countries_with_valid_responses  Country_coverage_percentage\n             Q57           General and interpersonal trust      737      310        0        0      290          0                1337                 1337                1.38              95883                              66                       100.00\n             Q58           General and interpersonal trust      102      125        0        0      105          0                 332                  332                0.34              96888                              66                       100.00\n             Q59           General and interpersonal trust      534      184        0        0      109          0                 827                  827                0.85              96393                              66  

In [46]:
q57_q105_classification.to_csv(
    "../WVS_analysis_output/WVS_Q57_Q105_feature_classification.csv",
    index=False
)

In [47]:
output_file = (
    "../WVS_analysis_output/"
    "WVS_Q57_Q105_feature_classification.csv"
)

q57_q105_classification.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)
print("Total classified variables:",
      q57_q105_classification["Variable"].nunique())

Saved: ../WVS_analysis_output/WVS_Q57_Q105_feature_classification.csv
Total classified variables: 77


In [48]:
# Check duplicate variable names
duplicates = q57_q105_classification[
    q57_q105_classification["Variable"].duplicated(
        keep=False
    )
]

print("Duplicate variables:", len(duplicates))

# Check total number of variables
print(
    "Unique variables:",
    q57_q105_classification["Variable"].nunique()
)

Duplicate variables: 0
Unique variables: 77


In [49]:
# Extract variables from Q106 to Q163
q106_q163 = remaining_inventory[
    remaining_inventory["Question_number"].between(106, 163)
].copy()

print("Number of variables:", len(q106_q163))
print("\nVariables:")
print(q106_q163["Variable"].tolist())

Number of variables: 58

Variables:
['Q106', 'Q107', 'Q108', 'Q109', 'Q110', 'Q111', 'Q112', 'Q113', 'Q114', 'Q115', 'Q116', 'Q117', 'Q118', 'Q119', 'Q120', 'Q121', 'Q122', 'Q123', 'Q124', 'Q125', 'Q126', 'Q127', 'Q128', 'Q129', 'Q130', 'Q131', 'Q132', 'Q133', 'Q134', 'Q135', 'Q136', 'Q137', 'Q138', 'Q139', 'Q140', 'Q141', 'Q142', 'Q143', 'Q144', 'Q145', 'Q146', 'Q147', 'Q148', 'Q149', 'Q150', 'Q151', 'Q152', 'Q153', 'Q154', 'Q155', 'Q156', 'Q157', 'Q158', 'Q159', 'Q160', 'Q161', 'Q162', 'Q163']


In [50]:
q106_q163_quality = (
    q106_q163[["Variable"]]
    .merge(
        remaining_quality.rename_axis("Variable").reset_index(),
        on="Variable",
        how="left"
    )
    .merge(
        remaining_coverage,
        on="Variable",
        how="left"
    )
)

print(
    q106_q163_quality[
        [
            "Variable",
            "Missing_percentage",
            "Countries_with_valid_responses",
            "Country_coverage_percentage",
            "Valid_respondents"
        ]
    ].to_string(index=False)
)

Variable  Missing_percentage  Countries_with_valid_responses  Country_coverage_percentage  Valid_respondents
    Q106                1.41                              66                       100.00              95851
    Q107                3.54                              66                       100.00              93776
    Q108                1.21                              66                       100.00              96042
    Q109                1.71                              66                       100.00              95558
    Q110                1.48                              66                       100.00              95779
    Q111                4.78                              66                       100.00              92569
    Q112                1.69                              66                       100.00              95575
    Q113                6.78                              64                        96.97              90631
    Q114           

In [51]:
variable_groups_106_163 = {
    "Economic Values": [
        f"Q{i}" for i in range(106, 112)
    ],

    "Perceived Corruption": [
        f"Q{i}" for i in range(112, 121)
    ],

    "Immigration Attitudes": [
        f"Q{i}" for i in range(121, 131)
    ],

    "Personal and Neighborhood Security": [
        f"Q{i}" for i in range(131, 142)
    ],

    "Security Concerns and Victimization": [
        f"Q{i}" for i in range(142, 149)
    ],

    "Freedom, Equality and Political Values": [
        f"Q{i}" for i in range(149, 152)
    ],

    "Postmaterialist Priorities": [
        f"Q{i}" for i in range(152, 158)
    ],

    "Science and Technology Attitudes": [
        f"Q{i}" for i in range(158, 164)
    ]
}

# Keep only variables that exist in the dataset
variable_groups_106_163 = {
    group: [
        variable for variable in variables
        if variable in survey_variable_names
    ]
    for group, variables in variable_groups_106_163.items()
}

# Print group sizes
for group, variables in variable_groups_106_163.items():
    print(f"{group}: {len(variables)} variables")

Economic Values: 6 variables
Perceived Corruption: 9 variables
Immigration Attitudes: 10 variables
Personal and Neighborhood Security: 11 variables
Security Concerns and Victimization: 7 variables
Freedom, Equality and Political Values: 3 variables
Postmaterialist Priorities: 6 variables
Science and Technology Attitudes: 6 variables


In [52]:
classification_rows_106_163 = []

for group, variables in variable_groups_106_163.items():
    for variable in variables:
        classification_rows_106_163.append({
            "Variable": variable,
            "Feature group": group
        })

q106_q163_classification = pd.DataFrame(
    classification_rows_106_163
)

q106_q163_classification = (
    q106_q163_classification
    .merge(
        q106_q163_quality,
        on="Variable",
        how="left"
    )
)

# Check duplicates
print(
    "Unique variables:",
    q106_q163_classification["Variable"].nunique()
)

print(
    "Duplicate variables:",
    q106_q163_classification["Variable"].duplicated().sum()
)

# Save
q106_q163_classification.to_csv(
    "../WVS_analysis_output/"
    "WVS_Q106_Q163_feature_classification.csv",
    index=False
)

print("Classification saved successfully.")

Unique variables: 58
Duplicate variables: 0
Classification saved successfully.


In [53]:
q164_q220 = remaining_inventory[
    remaining_inventory["Question_number"].between(164, 220)
].copy()

print("Number of variables:", len(q164_q220))
print("\nVariables:")
print(q164_q220["Variable"].tolist())

Number of variables: 58

Variables:
['Q164', 'Q165', 'Q166', 'Q167', 'Q168', 'Q169', 'Q170', 'Q171', 'Q172', 'Q172R', 'Q173', 'Q174', 'Q175', 'Q176', 'Q177', 'Q178', 'Q179', 'Q180', 'Q181', 'Q182', 'Q183', 'Q184', 'Q185', 'Q186', 'Q187', 'Q188', 'Q189', 'Q190', 'Q191', 'Q192', 'Q193', 'Q194', 'Q195', 'Q196', 'Q197', 'Q198', 'Q199', 'Q200', 'Q201', 'Q202', 'Q203', 'Q204', 'Q205', 'Q206', 'Q207', 'Q208', 'Q209', 'Q210', 'Q211', 'Q212', 'Q213', 'Q214', 'Q215', 'Q216', 'Q217', 'Q218', 'Q219', 'Q220']


In [54]:
q164_q220_quality = (
    q164_q220[["Variable"]]
    .merge(
        remaining_quality.rename_axis("Variable").reset_index(),
        on="Variable",
        how="left"
    )
    .merge(
        remaining_coverage,
        on="Variable",
        how="left"
    )
)

print(
    q164_q220_quality[
        [
            "Variable",
            "Missing_percentage",
            "Countries_with_valid_responses",
            "Country_coverage_percentage",
            "Valid_respondents"
        ]
    ].to_string(index=False)
)

Variable  Missing_percentage  Countries_with_valid_responses  Country_coverage_percentage  Valid_respondents
    Q164                2.32                              65                        98.48              94967
    Q165                4.83                              64                        96.97              92529
    Q166                7.81                              64                        96.97              89625
    Q167                7.35                              64                        96.97              90076
    Q168                7.17                              64                        96.97              90249
    Q169                5.75                              66                       100.00              91627
    Q170                7.53                              65                        98.48              89901
    Q171                1.25                              66                       100.00              96008
    Q172           

In [55]:
q164_q220_groups = {
    "Religious Beliefs": [
        f"Q{i}" for i in range(164, 171)
    ],

    "Religious Practices and Identity": [
        "Q171", "Q172", "Q172R",
        "Q173", "Q174", "Q175"
    ],

    "Ethical Values and Moral Norms": [
        f"Q{i}" for i in range(176, 196)
    ],

    "Government Surveillance Attitudes": [
        "Q196", "Q197", "Q198"
    ],

    "Political Interest and Discussion": [
        "Q199", "Q200"
    ],

    "News and Information Sources": [
        f"Q{i}" for i in range(201, 209)
    ],

    "Conventional Political Participation": [
        f"Q{i}" for i in range(209, 213)
    ],

    "Civic and Political Activism": [
        f"Q{i}" for i in range(213, 217)
    ],

    "Online Political Participation": [
        f"Q{i}" for i in range(217, 221)
    ]
}

In [56]:
classification_rows = []

for group_name, variables in q164_q220_groups.items():
    for variable in variables:
        classification_rows.append({
            "Variable": variable,
            "Feature_group": group_name
        })

q164_q220_classification = pd.DataFrame(classification_rows)

print(
    q164_q220_classification
    .groupby("Feature_group")
    .size()
    .sort_values(ascending=False)
)

Feature_group
Ethical Values and Moral Norms          20
News and Information Sources             8
Religious Beliefs                        7
Religious Practices and Identity         6
Civic and Political Activism             4
Online Political Participation           4
Conventional Political Participation     4
Government Surveillance Attitudes        3
Political Interest and Discussion        2
dtype: int64


In [57]:
classified_variables = q164_q220_classification["Variable"].tolist()

expected_variables = set(q164_q220["Variable"])
classified_set = set(classified_variables)

print("Expected variables:", len(expected_variables))
print("Classified variables:", len(classified_variables))
print("Unique variables:", len(classified_set))

print(
    "Missing variables:",
    sorted(expected_variables - classified_set)
)

print(
    "Unexpected variables:",
    sorted(classified_set - expected_variables)
)

print(
    "Duplicate variables:",
    q164_q220_classification[
        q164_q220_classification["Variable"].duplicated()
    ]["Variable"].tolist()
)

Expected variables: 58
Classified variables: 58
Unique variables: 58
Missing variables: []
Unexpected variables: []
Duplicate variables: []


In [58]:
q164_q220_final = (
    q164_q220_classification
    .merge(
        q164_q220_quality,
        on="Variable",
        how="left"
    )
)

output_file = (
    "../WVS_analysis_output/"
    "WVS_Q164_Q220_feature_classification.csv"
)

q164_q220_final.to_csv(
    output_file,
    index=False
)

print("Classification saved successfully.")

display(q164_q220_final.head(10))

Classification saved successfully.


,Variable,Feature_group,Code_-1,Code_-2,Code_-3,Code_-4,Code_-5,NaN_count,Special_code_count,Total_missing_count,Missing_percentage,Valid_respondents,Countries_with_valid_responses,Country_coverage_percentage
0,Q164,Religious Beliefs,490,356,0,1200,207,0,2253,2253,2.32,94967,65,98.48
1,Q165,Religious Beliefs,1607,465,0,2400,219,0,4691,4691,4.83,92529,64,96.97
2,Q166,Religious Beliefs,4205,776,0,2400,214,0,7595,7595,7.81,89625,64,96.97
3,Q167,Religious Beliefs,3694,805,0,2400,245,0,7144,7144,7.35,90076,64,96.97
4,Q168,Religious Beliefs,3567,761,0,2400,243,0,6971,6971,7.17,90249,64,96.97
5,Q169,Religious Beliefs,4079,1270,0,0,244,0,5593,5593,5.75,91627,66,100.00
6,Q170,Religious Beliefs,2777,2000,0,1200,1342,0,7319,7319,7.53,89901,65,98.48
7,Q171,Religious Practices and Identity,374,512,0,0,326,0,1212,1212,1.25,96008,66,100.00
8,Q172,Religious Practices and Identity,628,624,0,0,243,0,1495,1495,1.54,95725,66,100.00
9,Q172R,Religious Practices and Identity,617,660,0,0,216,0,1493,1493,1.54,95727,66,100.00


In [59]:
q221_onwards = remaining_inventory[
    remaining_inventory["Question_number"] >= 221
].copy()

print("Number of variables:", len(q221_onwards))
print("\nVariables:")
print(q221_onwards["Variable"].tolist())

Number of variables: 119

Variables:
['Q221', 'Q222', 'Q223', 'Q223_ABREV', 'Q223_LOCAL', 'Q224', 'Q225', 'Q226', 'Q227', 'Q228', 'Q229', 'Q230', 'Q231', 'Q232', 'Q233', 'Q234', 'Q234A', 'Q235', 'Q236', 'Q237', 'Q238', 'Q239', 'Q240', 'Q241', 'Q242', 'Q243', 'Q244', 'Q245', 'Q246', 'Q247', 'Q248', 'Q249', 'Q250', 'Q251', 'Q252', 'Q253', 'Q254', 'Q255', 'Q256', 'Q257', 'Q258', 'Q259', 'Q260', 'Q261', 'Q262', 'Q263', 'Q264', 'Q265', 'Q266', 'Q267', 'Q268', 'Q269', 'Q270', 'Q271', 'Q272', 'Q273', 'Q274', 'Q275', 'Q275A', 'Q275R', 'Q276', 'Q276A', 'Q276R', 'Q277', 'Q277A', 'Q277R', 'Q278', 'Q278A', 'Q278R', 'Q279', 'Q280', 'Q281', 'Q282', 'Q283', 'Q284', 'Q285', 'Q286', 'Q287', 'Q288', 'Q288R', 'Q289', 'Q289CS9', 'Q290', 'Q291G1', 'Q291G2', 'Q291G3', 'Q291G4', 'Q291G5', 'Q291G6', 'Q291P1', 'Q291P2', 'Q291P3', 'Q291P4', 'Q291P5', 'Q291P6', 'Q291UN1', 'Q291UN2', 'Q291UN3', 'Q291UN4', 'Q291UN5', 'Q291UN6', 'Q292A', 'Q292B', 'Q292C', 'Q292D', 'Q292E', 'Q292F', 'Q292G', 'Q292H', 'Q292I', 'Q292J

In [60]:
q221_onwards_quality = (
    q221_onwards[["Variable"]]
    .merge(
        remaining_quality.rename_axis("Variable").reset_index(),
        on="Variable",
        how="left"
    )
    .merge(
        remaining_coverage,
        on="Variable",
        how="left"
    )
)

display(
    q221_onwards_quality[
        [
            "Variable",
            "Missing_percentage",
            "Countries_with_valid_responses",
            "Country_coverage_percentage",
            "Valid_respondents"
        ]
    ].to_string(index=False)
)

'  Variable  Missing_percentage  Countries_with_valid_responses  Country_coverage_percentage  Valid_respondents\n      Q221                4.65                              64                        96.97              92700\n      Q222                5.57                              64                        96.97              91808\n      Q223               25.49                              62                        93.94              72441\nQ223_ABREV               25.49                              62                        93.94              72441\nQ223_LOCAL               25.49                              62                        93.94              72441\n      Q224                9.19                              65                        98.48              88287\n      Q225               13.54                              65                        98.48              84058\n      Q226               15.30                              61                        92.42            

In [61]:
display(
    q221_onwards_quality
    .sort_values("Missing_percentage", ascending=False)
    [
        [
            "Variable",
            "Missing_percentage",
            "Countries_with_valid_responses",
            "Valid_respondents"
        ]
    ]
    .head(20)
)

,Variable,Missing_percentage,Countries_with_valid_responses,Valid_respondents
117,Q294A,92.61,12,7182
118,Q294B,92.57,12,7224
116,Q293,87.56,10,12092
109,Q292I,84.19,13,15369
110,Q292J,84.14,13,15421
112,Q292L,84.11,13,15453
103,Q292C,84.11,13,15446
108,Q292H,84.10,13,15456
113,Q292M,84.09,13,15465
107,Q292G,84.08,13,15474


In [62]:
q221_q294_groups = {

    "Political Participation and Party Preference": [
        "Q221", "Q222", "Q223",
        "Q223_ABREV", "Q223_LOCAL"
    ],

    "Electoral Integrity and Political Voice": [
        *[f"Q{i}" for i in range(224, 235)],
        "Q234A"
    ],

    "Political Culture and Democratic Preferences": [
        f"Q{i}" for i in range(235, 260)
    ],

    "Basic Demographic Background": [
        f"Q{i}" for i in range(260, 275)
    ],

    "Educational Background": [
        "Q275", "Q275A", "Q275R",
        "Q276", "Q276A", "Q276R",
        "Q277", "Q277A", "Q277R",
        "Q278", "Q278A", "Q278R"
    ],

    "Employment and Occupational Background": [
        f"Q{i}" for i in range(279, 285)
    ],

    "Household Economic Position": [
        "Q285", "Q286", "Q287",
        "Q288", "Q288R"
    ],

    "Religious and Ethnic Identity": [
        "Q289", "Q289CS9", "Q290"
    ],

    "Political Trust - Parliament, Government and UN": [
        *[f"Q291P{i}" for i in range(1, 7)],
        *[f"Q291G{i}" for i in range(1, 7)],
        *[f"Q291UN{i}" for i in range(1, 7)]
    ],

    "Political Trust and Government Attitudes": [
        f"Q292{letter}" for letter in "ABCDEFGHIJKLMNO"
    ],

    "Trust in Political Leaders": [
        "Q293", "Q294A", "Q294B"
    ]
}

In [63]:
classification_rows = []

for group_name, variables in q221_q294_groups.items():

    for variable in variables:

        classification_rows.append({
            "Variable": variable,
            "Feature_group": group_name
        })

q221_q294_classification = pd.DataFrame(
    classification_rows
)

print(
    q221_q294_classification
    .groupby("Feature_group")
    .size()
    .sort_values(ascending=False)
)

Feature_group
Political Culture and Democratic Preferences       25
Political Trust - Parliament, Government and UN    18
Basic Demographic Background                       15
Political Trust and Government Attitudes           15
Educational Background                             12
Electoral Integrity and Political Voice            12
Employment and Occupational Background              6
Political Participation and Party Preference        5
Household Economic Position                         5
Religious and Ethnic Identity                       3
Trust in Political Leaders                          3
dtype: int64


In [64]:
classified_variables = (
    q221_q294_classification["Variable"].tolist()
)

expected_variables = set(q221_onwards["Variable"])
classified_set = set(classified_variables)

print("Expected variables:", len(expected_variables))
print("Classified variables:", len(classified_variables))
print("Unique variables:", len(classified_set))

print(
    "Missing variables:",
    sorted(expected_variables - classified_set)
)

print(
    "Unexpected variables:",
    sorted(classified_set - expected_variables)
)

print(
    "Duplicate variables:",
    q221_q294_classification[
        q221_q294_classification["Variable"].duplicated()
    ]["Variable"].tolist()
)

Expected variables: 119
Classified variables: 119
Unique variables: 119
Missing variables: []
Unexpected variables: []
Duplicate variables: []


In [65]:
q221_q294_final = (
    q221_q294_classification
    .merge(
        q221_onwards_quality,
        on="Variable",
        how="left"
    )
)

output_file = (
    "../WVS_analysis_output/"
    "WVS_Q221_Q294_feature_classification.csv"
)

q221_q294_final.to_csv(
    output_file,
    index=False
)

print("Classification saved successfully.")

display(q221_q294_final.head(10))

Classification saved successfully.


,Variable,Feature_group,Code_-1,Code_-2,Code_-3,Code_-4,Code_-5,NaN_count,Special_code_count,Total_missing_count,Missing_percentage,Valid_respondents,Countries_with_valid_responses,Country_coverage_percentage
0,Q221,Political Participation and Party Preference,554,411,0,3288,267,0,4520,4520,4.65,92700,64,96.97
1,Q222,Political Participation and Party Preference,573,475,0,4059,305,0,5412,5412,5.57,91808,64,96.97
2,Q223,Political Participation and Party Preference,10193,5974,629,6248,1735,0,24779,24779,25.49,72441,62,93.94
3,Q223_ABREV,Political Participation and Party Preference,10193,5974,629,6248,1735,0,24779,24779,25.49,72441,62,93.94
4,Q223_LOCAL,Political Participation and Party Preference,10193,5974,629,6248,1735,0,24779,24779,25.49,72441,62,93.94
5,Q224,Electoral Integrity and Political Voice,4820,853,0,3036,224,0,8933,8933,9.19,88287,65,98.48
6,Q225,Electoral Integrity and Political Voice,8550,1273,0,3036,303,0,13162,13162,13.54,84058,65,98.48
7,Q226,Electoral Integrity and Political Voice,5829,941,0,7839,264,0,14873,14873,15.30,82347,61,92.42
8,Q227,Electoral Integrity and Political Voice,6675,1031,0,3036,289,0,11031,11031,11.35,86189,65,98.48
9,Q228,Electoral Integrity and Political Voice,5952,962,0,3036,266,0,10216,10216,10.51,87004,65,98.48


## 12. Combine all feature classifications

This section combines the variable classifications from all
questionnaire sections into one master table.

The purpose is to create a complete overview of candidate
features before exploratory data analysis.

In [66]:
# Combine all feature classification tables

classification_tables = [
    q1_q56_classification,
    q57_q105_classification,
    q106_q163_classification,
    q164_q220_classification,
    q221_q294_classification
]

feature_classification_master = pd.concat(
    classification_tables,
    ignore_index=True
)

print("Total classified variables:",
      len(feature_classification_master))

print("Unique variables:",
      feature_classification_master["Variable"].nunique())

print("Duplicate variables:",
      feature_classification_master[
          feature_classification_master["Variable"].duplicated()
      ]["Variable"].tolist())

NameError: name 'q1_q56_classification' is not defined

In [68]:
import os
import glob

# 1. 查看当前 Notebook 中已有的分类变量
print("Classification variables in memory:")

current_variables = list(globals().keys())

for name in current_variables:
    if "classification" in name.lower():
        print(name)

# 2. 查看输出文件夹中已保存的分类 CSV
print("\nClassification CSV files:")

classification_files = glob.glob(
    "../WVS_analysis_output/*classification*.csv"
)

for file in classification_files:
    print(os.path.basename(file))

Classification variables in memory:
classification_rows
q57_q105_classification
classification_rows_106_163
q106_q163_classification
q164_q220_classification
q221_q294_classification

Classification CSV files:
WVS_Q106_Q163_feature_classification.csv
WVS_Q164_Q220_feature_classification.csv
WVS_Q221_Q294_feature_classification.csv
WVS_Q57_Q105_conceptual_classification.csv
WVS_Q57_Q105_feature_classification.csv


In [69]:
# 查看之前可能用于 Q1–Q56 分类的变量
print("Checking possible Q1-Q56 classification variables:")

possible_names = [
    name for name in list(globals().keys())
    if any(keyword in name.lower() for keyword in [
        "candidate",
        "q1_q56",
        "q1",
        "feature_group"
    ])
]

for name in possible_names:
    obj = globals()[name]

    if isinstance(obj, pd.DataFrame):
        print(f"\n{name}:")
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())

Checking possible Q1-Q56 classification variables:

candidate_feature_table:
Shape: (55, 19)
Columns: ['Column number', 'Variable', 'Data type', 'Missing count (NaN)', 'Missing percentage (NaN)', 'Unique values (excluding NaN)', 'Candidate special-code count', 'Candidate special-code details', 'Example values', 'Preliminary category', 'Code_-1', 'Code_-2', 'Code_-3', 'Code_-4', 'Code_-5', 'NaN_count', 'Special_code_count', 'Total_missing_count', 'Missing_percentage']

candidate_screening_table:
Shape: (55, 6)
Columns: ['Variable', 'Missing_percentage', 'Countries_with_valid_responses', 'Country_coverage_percentage', 'Respondents_with_valid_responses', 'Measurement_type']

q57_q105:
Shape: (78, 12)
Columns: ['Column number', 'Variable', 'Data type', 'Missing count (NaN)', 'Missing percentage (NaN)', 'Unique values (excluding NaN)', 'Candidate special-code count', 'Candidate special-code details', 'Example values', 'Preliminary category', 'Variable_pattern', 'Question_number']

q57_q105_

In [70]:
# Define conceptual groups for Q1-Q56
# Q49 is the target variable and must not be included as a predictor.

q1_q56_groups = {
    "Personal Values": list(range(1, 7)),
    "Child-rearing Values": list(range(7, 18)),
    "Social Tolerance": list(range(18, 27)),
    "Family and Gender Norms": list(range(27, 39)),
    "Work and Social Attitudes": list(range(39, 46)),
    "Wellbeing": [46, 47, 48],
    "Economic Security": list(range(50, 57))
}

classification_rows = []

for group, question_numbers in q1_q56_groups.items():
    for number in question_numbers:
        classification_rows.append({
            "Variable": f"Q{number}",
            "Feature_group": group
        })

q1_q56_classification = pd.DataFrame(classification_rows)

print("Classified variables:", len(q1_q56_classification))
print("Unique variables:", q1_q56_classification["Variable"].nunique())

print("\nVariables missing from classification:")
print(
    sorted(
        set(candidate_feature_table["Variable"])
        - set(q1_q56_classification["Variable"])
    )
)

print("\nVariables not found in candidate table:")
print(
    sorted(
        set(q1_q56_classification["Variable"])
        - set(candidate_feature_table["Variable"])
    )
)

display(q1_q56_classification.groupby("Feature_group").size())

Classified variables: 55
Unique variables: 55

Variables missing from classification:
[]

Variables not found in candidate table:
[]


Feature_group
Child-rearing Values         11
Economic Security             7
Family and Gender Norms      12
Personal Values               6
Social Tolerance              9
Wellbeing                     3
Work and Social Attitudes     7
dtype: int64

In [71]:
# Standardize column names before combining

classification_tables = [
    q1_q56_classification,
    q57_q105_classification,
    q106_q163_classification,
    q164_q220_classification,
    q221_q294_classification
]

standardized_tables = []

for table in classification_tables:
    temp = table.copy()

    # Standardize the feature-group column name
    temp = temp.rename(columns={
        "Feature group": "Feature_group"
    })

    # Keep only the two columns needed for classification
    temp = temp[["Variable", "Feature_group"]]

    standardized_tables.append(temp)

master_classification = pd.concat(
    standardized_tables,
    ignore_index=True
)

print("Total classified rows:", len(master_classification))

print(
    "Unique classified variables:",
    master_classification["Variable"].nunique()
)

print(
    "Duplicate variables:",
    master_classification.loc[
        master_classification["Variable"].duplicated(),
        "Variable"
    ].tolist()
)

Total classified rows: 367
Unique classified variables: 367
Duplicate variables: []


In [72]:
# Compare classification against the original survey variable inventory

all_survey_variables = set(survey_variable_names)

classified_variables = set(
    master_classification["Variable"]
)

unclassified_variables = sorted(
    all_survey_variables - classified_variables
)

unexpected_variables = sorted(
    classified_variables - all_survey_variables
)

print("Survey variables in inventory:", len(all_survey_variables))
print("Classified variables:", len(classified_variables))

print("\nUnclassified variables:")
print(unclassified_variables)

print("\nUnexpected variables:")
print(unexpected_variables)

Survey variables in inventory: 372
Classified variables: 367

Unclassified variables:
['Q33_3', 'Q34_3', 'Q35_3', 'Q49', 'Q82']

Unexpected variables:
[]


In [73]:
# Inspect the five unclassified variables

unclassified_variables = [
    "Q33_3",
    "Q34_3",
    "Q35_3",
    "Q49",
    "Q82"
]

for variable in unclassified_variables:
    print(f"\n{'=' * 40}")
    print("Variable:", variable)

    if variable in survey_data.columns:
        print("Unique values:")
        print(
            survey_data[variable]
            .value_counts(dropna=False)
            .head(15)
        )
    else:
        print("Not found in survey_data")


Variable: Q33_3
Unique values:
Q33_3
 2    44494
 1    37817
 3    14173
-1      503
-2      122
-5      111
Name: count, dtype: int64

Variable: Q34_3
Unique values:
Q34_3
 1    66788
 2    17232
 3    12287
-1      626
-2      177
-5      110
Name: count, dtype: int64

Variable: Q35_3
Unique values:
Q35_3
 2    42906
 1    33429
 3    19421
-1     1140
-2      199
-5      125
Name: count, dtype: int64

Variable: Q49
Unique values:
Q49
 8     19689
 7     16377
 10    16352
 5     11530
 6     10986
 9     10097
 4      4476
 3      3140
 1      2632
 2      1421
-1       205
-2       203
-5       112
Name: count, dtype: int64

Variable: Q82
Unique values:
Q82
 3    26836
 2    25914
 4    15249
-1    11086
-4     9352
 1     7683
-2      893
-5      207
Name: count, dtype: int64


In [74]:
# Classify the five previously unclassified variables

special_classification = pd.DataFrame([
    {
        "Variable": "Q33_3",
        "Feature_group": "Work and Social Attitudes",
        "Variable_role": "Alternative version",
        "Model_status": "Review before modeling"
    },
    {
        "Variable": "Q34_3",
        "Feature_group": "Work and Social Attitudes",
        "Variable_role": "Alternative version",
        "Model_status": "Review before modeling"
    },
    {
        "Variable": "Q35_3",
        "Feature_group": "Work and Social Attitudes",
        "Variable_role": "Alternative version",
        "Model_status": "Review before modeling"
    },
    {
        "Variable": "Q49",
        "Feature_group": "Life Satisfaction",
        "Variable_role": "Target",
        "Model_status": "Target only; exclude from X"
    },
    {
        "Variable": "Q82",
        "Feature_group": "Institutional Confidence",
        "Variable_role": "Predictor candidate",
        "Model_status": "Keep for exploratory analysis"
    }
])

display(special_classification)

,Variable,Feature_group,Variable_role,Model_status
0,Q33_3,Work and Social Attitudes,Alternative version,Review before modeling
1,Q34_3,Work and Social Attitudes,Alternative version,Review before modeling
2,Q35_3,Work and Social Attitudes,Alternative version,Review before modeling
3,Q49,Life Satisfaction,Target,Target only; exclude from X
4,Q82,Institutional Confidence,Predictor candidate,Keep for exploratory analysis


In [78]:
# Add role information to the existing master classification

master_classification["Variable_role"] = "Predictor candidate"
master_classification["Model_status"] = "Candidate"

# Add the five special variables
master_classification = pd.concat(
    [
        master_classification,
        special_classification
    ],
    ignore_index=True
)

# Check completeness
print("Total rows:", len(master_classification))
print(
    "Unique variables:",
    master_classification["Variable"].nunique()
)

print(
    "Duplicate variables:",
    master_classification.loc[
        master_classification["Variable"].duplicated(),
        "Variable"
    ].tolist()
)

print("\nRole distribution:")
display(
    master_classification["Variable_role"].value_counts()
)

Total rows: 377
Unique variables: 372
Duplicate variables: ['Q33_3', 'Q34_3', 'Q35_3', 'Q49', 'Q82']

Role distribution:


Variable_role
Predictor candidate    373
Alternative version      3
Target                   1
Name: count, dtype: int64

In [76]:
# Save the master classification table

output_file = (
    "../WVS_analysis_output/"
    "WVS_Master_Feature_Classification.csv"
)

master_classification.to_csv(
    output_file,
    index=False
)

print("Master classification saved:")
print(output_file)

Master classification saved:
../WVS_analysis_output/WVS_Master_Feature_Classification.csv


In [77]:
# Correct the feature group for Q33_3, Q34_3 and Q35_3

alternative_variables = ["Q33_3", "Q34_3", "Q35_3"]

master_classification.loc[
    master_classification["Variable"].isin(alternative_variables),
    "Feature_group"
] = "Family and Gender Norms"

# Verify the correction
display(
    master_classification[
        master_classification["Variable"].isin(alternative_variables)
    ]
)

,Variable,Feature_group,Variable_role,Model_status
367,Q33_3,Family and Gender Norms,Alternative version,Review before modeling
368,Q34_3,Family and Gender Norms,Alternative version,Review before modeling
369,Q35_3,Family and Gender Norms,Alternative version,Review before modeling


In [79]:
output_file = (
    "../WVS_analysis_output/"
    "WVS_Master_Feature_Classification.csv"
)

master_classification.to_csv(output_file, index=False)

print("Master table updated and saved.")

Master table updated and saved.


In [80]:
# Display the Q164-Q220 classification

display(
    q164_q220_classification[
        ["Variable", "Feature_group"]
    ].sort_values("Variable")
)

,Variable,Feature_group
0,Q164,Religious Beliefs
1,Q165,Religious Beliefs
2,Q166,Religious Beliefs
3,Q167,Religious Beliefs
4,Q168,Religious Beliefs
5,Q169,Religious Beliefs
6,Q170,Religious Beliefs
7,Q171,Religious Practices and Identity
8,Q172,Religious Practices and Identity
9,Q172R,Religious Practices and Identity


## 8. Building the Master Variable Dictionary

This section combines the completed variable classification, 
initial measurement classification, and dataset inventory 
into one consolidated variable dictionary.

In [81]:
import pandas as pd
import numpy as np
from pathlib import Path

# Define the output directory
OUTPUT_DIR = Path("../WVS_analysis_output")

# Load the three existing files
master_classification = pd.read_csv(
    OUTPUT_DIR / "WVS_Master_Feature_Classification.csv"
)

variable_inventory = pd.read_csv(
    OUTPUT_DIR / "WVS_variable_inventory.csv"
)

measurement_dictionary = pd.read_csv(
    OUTPUT_DIR / "WVS_candidate_measurement_dictionary.csv"
)

print("Master classification:", master_classification.shape)
print("Variable inventory:", variable_inventory.shape)
print("Measurement dictionary:", measurement_dictionary.shape)

Master classification: (377, 4)
Variable inventory: (613, 10)
Measurement dictionary: (55, 8)


In [82]:
print("MASTER CLASSIFICATION")
print(master_classification.columns.tolist())

print("\nVARIABLE INVENTORY")
print(variable_inventory.columns.tolist())

print("\nMEASUREMENT DICTIONARY")
print(measurement_dictionary.columns.tolist())

MASTER CLASSIFICATION
['Variable', 'Feature_group', 'Variable_role', 'Model_status']

VARIABLE INVENTORY
['Column number', 'Variable', 'Data type', 'Missing count (NaN)', 'Missing percentage (NaN)', 'Unique values (excluding NaN)', 'Candidate special-code count', 'Candidate special-code details', 'Example values', 'Preliminary category']

MEASUREMENT DICTIONARY
['Variable', 'Missing_percentage', 'Countries_with_valid_responses', 'Country_coverage_percentage', 'Respondents_with_valid_responses', 'Measurement_type', 'Unique_value_count', 'Observed_values']


In [83]:
# Check total rows and unique variables
print("Total rows:", len(master_classification))

print(
    "Unique variables:",
    master_classification["Variable"].nunique()
)

print(
    "Duplicate variable names:",
    master_classification["Variable"].duplicated().sum()
)

# Display duplicated variables, if any
duplicates = master_classification[
    master_classification["Variable"].duplicated(keep=False)
].sort_values("Variable")

print("\nDuplicated variables:")
display(duplicates)

Total rows: 377
Unique variables: 372
Duplicate variable names: 5

Duplicated variables:


,Variable,Feature_group,Variable_role,Model_status
367,Q33_3,Family and Gender Norms,Predictor candidate,Candidate
372,Q33_3,Work and Social Attitudes,Alternative version,Review before modeling
368,Q34_3,Family and Gender Norms,Predictor candidate,Candidate
373,Q34_3,Work and Social Attitudes,Alternative version,Review before modeling
369,Q35_3,Family and Gender Norms,Predictor candidate,Candidate
374,Q35_3,Work and Social Attitudes,Alternative version,Review before modeling
370,Q49,Life Satisfaction,Predictor candidate,Candidate
375,Q49,Life Satisfaction,Target,Target only; exclude from X
371,Q82,Institutional Confidence,Predictor candidate,Candidate
376,Q82,Institutional Confidence,Predictor candidate,Keep for exploratory analysis


In [84]:
# Get unique variable names from each file
classification_vars = set(master_classification["Variable"])
inventory_vars = set(variable_inventory["Variable"])
measurement_vars = set(measurement_dictionary["Variable"])

# Variables in classification but not in inventory
classification_not_inventory = (
    classification_vars - inventory_vars
)

# Variables in inventory but not in classification
inventory_not_classification = (
    inventory_vars - classification_vars
)

# Variables in measurement dictionary but not in classification
measurement_not_classification = (
    measurement_vars - classification_vars
)

print(
    "Classification variables:",
    len(classification_vars)
)

print(
    "Inventory variables:",
    len(inventory_vars)
)

print(
    "Measurement dictionary variables:",
    len(measurement_vars)
)

print("\nClassification not in inventory:")
print(sorted(classification_not_inventory))

print("\nInventory not in classification:")
print(sorted(inventory_not_classification))

print("\nMeasurement not in classification:")
print(sorted(measurement_not_classification))

Classification variables: 372
Inventory variables: 613
Measurement dictionary variables: 55

Classification not in inventory:
[]

Inventory not in classification:
['AUTONOMY', 'A_STUDY', 'A_WAVE', 'A_YEAR', 'B_COUNTRY', 'B_COUNTRY_ALPHA', 'CHOICE', 'CPARTY', 'CPARTYABB', 'C_COW_ALPHA', 'C_COW_NUM', 'DEFIANCE', 'DGI', 'DISBELIEF', 'D_INTERVIEW', 'E1_LITERACY', 'EQUALITY', 'E_RESPINT', 'FW_END', 'FW_START', 'F_INTPRIVACY', 'GDPpercap1', 'GDPpercap2', 'GII', 'GPS_V10', 'GPS_V11', 'GPS_V12', 'GPS_V13', 'GPS_V14', 'GPS_V15', 'GPS_V16', 'GPS_V17', 'GPS_V4_Scale', 'GPS_V6_Scale', 'GPS_V8_Scale', 'GPS_V9', 'G_TOWNSIZE', 'G_TOWNSIZE2', 'H_SETTLEMENT', 'H_URBRURAL', 'ID_GPS', 'ID_PartyFacts', 'I_ABORTLIB', 'I_AUTHORITY', 'I_DEVOUT', 'I_DIVORLIB', 'I_HOMOLIB', 'I_IMAGIN', 'I_INDEP', 'I_NATIONALISM', 'I_NONOBED', 'I_NORM1', 'I_NORM2', 'I_NORM3', 'I_PSU', 'I_RELIGBEL', 'I_RELIGIMP', 'I_RELIGPRAC', 'I_TRUSTARMY', 'I_TRUSTCOURTS', 'I_TRUSTPOLICE', 'I_VOI2_00', 'I_VOICE1', 'I_VOICE2', 'I_WOMEDU', 'I_W

In [85]:
# Create a working copy
master_classification_clean = master_classification.copy()

# Correct Q33_3, Q34_3 and Q35_3
alternative_variables = ["Q33_3", "Q34_3", "Q35_3"]

for variable in alternative_variables:
    mask = master_classification_clean["Variable"] == variable

    master_classification_clean.loc[mask, "Feature_group"] = (
        "Family and Gender Norms"
    )

    master_classification_clean.loc[mask, "Variable_role"] = (
        "Alternative version"
    )

    master_classification_clean.loc[mask, "Model_status"] = (
        "Review before modeling"
    )


# Correct Q49: target variable only
mask = master_classification_clean["Variable"] == "Q49"

master_classification_clean.loc[mask, "Feature_group"] = (
    "Life Satisfaction"
)

master_classification_clean.loc[mask, "Variable_role"] = (
    "Target"
)

master_classification_clean.loc[mask, "Model_status"] = (
    "Target only; exclude from X"
)


# Correct Q82: exploratory analysis
mask = master_classification_clean["Variable"] == "Q82"

master_classification_clean.loc[mask, "Feature_group"] = (
    "Institutional Confidence"
)

master_classification_clean.loc[mask, "Variable_role"] = (
    "Predictor candidate"
)

master_classification_clean.loc[mask, "Model_status"] = (
    "Keep for exploratory analysis"
)


# Remove duplicate variable records
master_classification_clean = (
    master_classification_clean
    .drop_duplicates(subset="Variable", keep="first")
    .reset_index(drop=True)
)


print("Rows after cleaning:", len(master_classification_clean))

print(
    "Unique variables:",
    master_classification_clean["Variable"].nunique()
)

print(
    "Remaining duplicates:",
    master_classification_clean["Variable"].duplicated().sum()
)

Rows after cleaning: 372
Unique variables: 372
Remaining duplicates: 0


In [86]:
variables_to_check = [
    "Q33_3",
    "Q34_3",
    "Q35_3",
    "Q49",
    "Q82"
]

display(
    master_classification_clean[
        master_classification_clean["Variable"].isin(variables_to_check)
    ].sort_values("Variable")
)

,Variable,Feature_group,Variable_role,Model_status
367,Q33_3,Family and Gender Norms,Alternative version,Review before modeling
368,Q34_3,Family and Gender Norms,Alternative version,Review before modeling
369,Q35_3,Family and Gender Norms,Alternative version,Review before modeling
370,Q49,Life Satisfaction,Target,Target only; exclude from X
371,Q82,Institutional Confidence,Predictor candidate,Keep for exploratory analysis


In [87]:
category_mapping = {

    # 1. Economic
    "Economic Security": "Economic",
    "Economic Values": "Economic",
    "Household Economic Position": "Economic",

    # 2. Demographic & Background
    "Basic Demographic Background": "Demographic & Background",
    "Educational Background": "Demographic & Background",
    "Employment and Occupational Background": "Demographic & Background",
    "Religious and Ethnic Identity": "Demographic & Background",

    # 3. Social & Community
    "Child-rearing Values": "Social & Community",
    "Social Tolerance": "Social & Community",
    "Family and Gender Norms": "Social & Community",
    "General and Interpersonal Trust": "Social & Community",
    "Organizational Membership": "Social & Community",
    "Recoded Membership Variables": "Social & Community",
    "Immigration Attitudes": "Social & Community",
    "News and Information Sources": "Social & Community",

    # 4. Political & Institutional
    "Institutional Confidence": "Political & Institutional",
    "Country-specific Institutional Confidence": "Political & Institutional",
    "International Organization Attitudes": "Political & Institutional",
    "International Organization Knowledge": "Political & Institutional",
    "Perceived Corruption": "Political & Institutional",
    "Freedom, Equality and Political Values": "Political & Institutional",
    "Government Surveillance Attitudes": "Political & Institutional",
    "Political Interest and Discussion": "Political & Institutional",
    "Conventional Political Participation": "Political & Institutional",
    "Civic and Political Activism": "Political & Institutional",
    "Online Political Participation": "Political & Institutional",
    "Political Participation and Party Preference": "Political & Institutional",
    "Electoral Integrity and Political Voice": "Political & Institutional",
    "Political Culture and Democratic Preferences": "Political & Institutional",
    "Political Trust - Parliament, Government and UN": "Political & Institutional",
    "Political Trust and Government Attitudes": "Political & Institutional",
    "Trust in Political Leaders": "Political & Institutional",

    # 5. Personal Values & Wellbeing
    "Personal Values": "Personal Values & Wellbeing",
    "Work and Social Attitudes": "Personal Values & Wellbeing",
    "Wellbeing": "Personal Values & Wellbeing",
    "Religious Beliefs": "Personal Values & Wellbeing",
    "Religious Practices and Identity": "Personal Values & Wellbeing",
    "Ethical Values and Moral Norms": "Personal Values & Wellbeing",
    "Postmaterialist Priorities": "Personal Values & Wellbeing",

    # 6. Security
    "Personal and Neighborhood Security": "Security",
    "Security Concerns and Victimization": "Security",

    # 7. Science & Technology
    "Science and Technology Attitudes": "Science & Technology",

    # 8. Target Variable
    "Life Satisfaction": "Target Variable"
}

In [88]:
master_classification_clean["Category"] = (
    master_classification_clean["Feature_group"]
    .map(category_mapping)
)

# Check feature groups without a category
unmapped_groups = (
    master_classification_clean.loc[
        master_classification_clean["Category"].isna(),
        "Feature_group"
    ]
    .unique()
)

print("Unmapped feature groups:", unmapped_groups)

print("\nCategory distribution:")

display(
    master_classification_clean["Category"]
    .value_counts(dropna=False)
    .to_frame("Number of variables")
)

Unmapped feature groups: <StringArray>
[          'General and interpersonal trust',
                  'Institutional confidence',
 'Country-specific institutional confidence',
      'International organization attitudes',
      'International organization knowledge',
                 'Organizational membership',
              'Recoded membership variables']
Length: 7, dtype: str

Category distribution:


,Number of variables
Category,
Political & Institutional,108
NaN,77
Personal Values & Wellbeing,55
Social & Community,53
Demographic & Background,36
Economic,18
Security,18
Science & Technology,6
Target Variable,1


In [89]:
# Normalize feature group names
master_classification_clean["Feature_group_normalized"] = (
    master_classification_clean["Feature_group"]
    .str.strip()
    .str.casefold()
)

# Normalize category mapping keys
normalized_category_mapping = {
    key.strip().casefold(): value
    for key, value in category_mapping.items()
}

# Assign categories using normalized names
master_classification_clean["Category"] = (
    master_classification_clean["Feature_group_normalized"]
    .map(normalized_category_mapping)
)

# Check unmatched feature groups
unmapped_groups = (
    master_classification_clean.loc[
        master_classification_clean["Category"].isna(),
        "Feature_group"
    ]
    .unique()
)

print("Unmapped feature groups:", unmapped_groups)

print("\nCategory distribution:")

display(
    master_classification_clean["Category"]
    .value_counts(dropna=False)
    .to_frame("Number of variables")
)

Unmapped feature groups: <StringArray>
[]
Length: 0, dtype: str

Category distribution:


,Number of variables
Category,
Political & Institutional,154
Social & Community,84
Personal Values & Wellbeing,55
Demographic & Background,36
Economic,18
Security,18
Science & Technology,6
Target Variable,1


In [90]:
# Verify category completeness
total_variables = len(master_classification_clean)

categorized_variables = (
    master_classification_clean["Category"].notna().sum()
)

uncategorized_variables = (
    master_classification_clean["Category"].isna().sum()
)

print("Total variables:", total_variables)
print("Categorized variables:", categorized_variables)
print("Uncategorized variables:", uncategorized_variables)

assert total_variables == 372
assert categorized_variables == 372
assert uncategorized_variables == 0

print("\nAll variables have been successfully categorized.")

Total variables: 372
Categorized variables: 372
Uncategorized variables: 0

All variables have been successfully categorized.


In [91]:
category_summary = (
    master_classification_clean
    .groupby("Category")
    .agg(
        Variable_count=("Variable", "nunique"),
        Feature_group_count=("Feature_group", "nunique")
    )
    .sort_values("Variable_count", ascending=False)
)

display(category_summary)

,Variable_count,Feature_group_count
Category,,
Political & Institutional,154,18
Social & Community,84,8
Personal Values & Wellbeing,55,7
Demographic & Background,36,4
Economic,18,3
Security,18,2
Science & Technology,6,1
Target Variable,1,1


In [92]:
# Merge classification with inventory
master_variable_dictionary = master_classification_clean.merge(
    variable_inventory,
    on="Variable",
    how="left",
    validate="one_to_one"
)

print("Shape:", master_variable_dictionary.shape)
print("Unique variables:", master_variable_dictionary["Variable"].nunique())

Shape: (372, 15)
Unique variables: 372


In [93]:
# Merge measurement information
master_variable_dictionary = master_variable_dictionary.merge(
    measurement_dictionary[
        [
            "Variable",
            "Measurement_type",
            "Countries_with_valid_responses",
            "Country_coverage_percentage",
            "Respondents_with_valid_responses"
        ]
    ],
    on="Variable",
    how="left",
    validate="one_to_one"
)

print("Final shape:", master_variable_dictionary.shape)

print(
    "Variables with measurement information:",
    master_variable_dictionary["Measurement_type"].notna().sum()
)

print(
    "Variables without measurement information:",
    master_variable_dictionary["Measurement_type"].isna().sum()
)

Final shape: (372, 19)
Variables with measurement information: 55
Variables without measurement information: 317


In [94]:
display(
    master_variable_dictionary[
        [
            "Variable",
            "Category",
            "Feature_group",
            "Variable_role",
            "Measurement_type",
            "Data type",
            "Missing percentage (NaN)",
            "Country_coverage_percentage",
            "Model_status"
        ]
    ].head(20)
)

,Variable,Category,Feature_group,Variable_role,Measurement_type,Data type,Missing percentage (NaN),Country_coverage_percentage,Model_status
0,Q1,Personal Values & Wellbeing,Personal Values,Predictor candidate,Ordinal,int64,0.0,100.0,Candidate
1,Q2,Personal Values & Wellbeing,Personal Values,Predictor candidate,Ordinal,int64,0.0,100.0,Candidate
2,Q3,Personal Values & Wellbeing,Personal Values,Predictor candidate,Ordinal,int64,0.0,100.0,Candidate
3,Q4,Personal Values & Wellbeing,Personal Values,Predictor candidate,Ordinal,int64,0.0,100.0,Candidate
4,Q5,Personal Values & Wellbeing,Personal Values,Predictor candidate,Ordinal,int64,0.0,100.0,Candidate
5,Q6,Personal Values & Wellbeing,Personal Values,Predictor candidate,Ordinal,int64,0.0,100.0,Candidate
6,Q7,Social & Community,Child-rearing Values,Predictor candidate,Binary,int64,0.0,100.0,Candidate
7,Q8,Social & Community,Child-rearing Values,Predictor candidate,Binary,int64,0.0,100.0,Candidate
8,Q9,Social & Community,Child-rearing Values,Predictor candidate,Binary,int64,0.0,100.0,Candidate
9,Q10,Social & Community,Child-rearing Values,Predictor candidate,Binary,int64,0.0,100.0,Candidate


In [95]:
# Basic integrity checks
assert len(master_variable_dictionary) == 372

assert master_variable_dictionary["Variable"].nunique() == 372

assert master_variable_dictionary["Category"].notna().all()

assert (
    master_variable_dictionary["Variable"].is_unique
)

print("All integrity checks passed.")

All integrity checks passed.


In [96]:
measurement_coverage = (
    master_variable_dictionary["Measurement_type"]
    .notna()
    .value_counts()
)

print(measurement_coverage)

Measurement_type
False    317
True      55
Name: count, dtype: int64


In [97]:
output_file = (
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv"
)

master_variable_dictionary.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: ..\WVS_analysis_output\WVS_Master_Variable_Dictionary.csv


In [98]:
# Remove the temporary helper column
master_variable_dictionary = master_variable_dictionary.drop(
    columns=["Feature_group_normalized"],
    errors="ignore"
)

print("Final shape:", master_variable_dictionary.shape)

Final shape: (372, 18)


In [99]:
MASTER_DICTIONARY_FILE = (
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv"
)

master_variable_dictionary.to_csv(
    MASTER_DICTIONARY_FILE,
    index=False
)

print("Saved:", MASTER_DICTIONARY_FILE)

Saved: ..\WVS_analysis_output\WVS_Master_Variable_Dictionary.csv


## 11. Measurement Type Coverage Analysis

This section evaluates how many variables have verified measurement
types across the eight main analytical categories.

The purpose is to identify gaps in measurement classification and
prioritize variables for further review.

In [100]:
# Count variables with and without measurement information
measurement_coverage = (
    master_variable_dictionary
    .assign(
        Measurement_classified=lambda x:
            x["Measurement_type"].notna()
    )
    .groupby("Category")
    .agg(
        Total_variables=("Variable", "count"),
        Classified_variables=("Measurement_classified", "sum")
    )
    .reset_index()
)

# Calculate unclassified variables and coverage percentage
measurement_coverage["Unclassified_variables"] = (
    measurement_coverage["Total_variables"]
    - measurement_coverage["Classified_variables"]
)

measurement_coverage["Coverage_percentage"] = (
    measurement_coverage["Classified_variables"]
    / measurement_coverage["Total_variables"]
    * 100
)

measurement_coverage = measurement_coverage.sort_values(
    "Coverage_percentage",
    ascending=True
)

measurement_coverage

,Category,Total_variables,Classified_variables,Unclassified_variables,Coverage_percentage
0,Demographic & Background,36,0,36,0.000000
3,Political & Institutional,154,0,154,0.000000
5,Security,18,0,18,0.000000
4,Science & Technology,6,0,6,0.000000
7,Target Variable,1,0,1,0.000000
2,Personal Values & Wellbeing,55,16,39,29.090909
6,Social & Community,84,32,52,38.095238
1,Economic,18,7,11,38.888889


In [101]:
feature_group_coverage = (
    master_variable_dictionary
    .assign(
        Measurement_classified=lambda x:
            x["Measurement_type"].notna()
    )
    .groupby(
        ["Category", "Feature_group"]
    )
    .agg(
        Total_variables=("Variable", "count"),
        Classified_variables=("Measurement_classified", "sum")
    )
    .reset_index()
)

feature_group_coverage["Unclassified_variables"] = (
    feature_group_coverage["Total_variables"]
    - feature_group_coverage["Classified_variables"]
)

feature_group_coverage["Coverage_percentage"] = (
    feature_group_coverage["Classified_variables"]
    / feature_group_coverage["Total_variables"]
    * 100
)

feature_group_coverage = feature_group_coverage.sort_values(
    ["Coverage_percentage", "Total_variables"],
    ascending=[True, False]
)

feature_group_coverage

,Category,Feature_group,Total_variables,Classified_variables,Unclassified_variables,Coverage_percentage
21,Political & Institutional,Institutional confidence,25,0,25,0.0
26,Political & Institutional,Political Culture and Democratic Preferences,25,0,25,0.0
7,Personal Values & Wellbeing,Ethical Values and Moral Norms,20,0,20,0.0
29,Political & Institutional,"Political Trust - Parliament, Government and UN",18,0,18,0.0
16,Political & Institutional,Country-specific institutional confidence,17,0,17,0.0
0,Demographic & Background,Basic Demographic Background,15,0,15,0.0
30,Political & Institutional,Political Trust and Government Attitudes,15,0,15,0.0
1,Demographic & Background,Educational Background,12,0,12,0.0
17,Political & Institutional,Electoral Integrity and Political Voice,12,0,12,0.0
40,Social & Community,Organizational membership,12,0,12,0.0


In [102]:
unclassified_measurement_variables = (
    master_variable_dictionary[
        master_variable_dictionary["Measurement_type"].isna()
    ][
        [
            "Variable",
            "Category",
            "Feature_group",
            "Variable_role",
            "Model_status",
            "Data type",
            "Missing percentage (NaN)",
            "Country_coverage_percentage"
        ]
    ]
    .sort_values(
        ["Category", "Feature_group", "Variable"]
    )
    .reset_index(drop=True)
)

print(
    "Variables requiring measurement classification:",
    len(unclassified_measurement_variables)
)

unclassified_measurement_variables.head(30)

Variables requiring measurement classification: 317


,Variable,Category,Feature_group,Variable_role,Model_status,Data type,Missing percentage (NaN),Country_coverage_percentage
0,Q260,Demographic & Background,Basic Demographic Background,Predictor candidate,Candidate,int64,0.0,NaN
1,Q261,Demographic & Background,Basic Demographic Background,Predictor candidate,Candidate,int64,0.0,NaN
2,Q262,Demographic & Background,Basic Demographic Background,Predictor candidate,Candidate,int64,0.0,NaN
3,Q263,Demographic & Background,Basic Demographic Background,Predictor candidate,Candidate,int64,0.0,NaN
4,Q264,Demographic & Background,Basic Demographic Background,Predictor candidate,Candidate,int64,0.0,NaN
5,Q265,Demographic & Background,Basic Demographic Background,Predictor candidate,Candidate,int64,0.0,NaN
6,Q266,Demographic & Background,Basic Demographic Background,Predictor candidate,Candidate,int64,0.0,NaN
7,Q267,Demographic & Background,Basic Demographic Background,Predictor candidate,Candidate,int64,0.0,NaN
8,Q268,Demographic & Background,Basic Demographic Background,Predictor candidate,Candidate,int64,0.0,NaN
9,Q269,Demographic & Background,Basic Demographic Background,Predictor candidate,Candidate,int64,0.0,NaN


## 14. Demographic and Background Measurement Classification

This section examines the measurement scales of demographic,
educational, employment, household and identity variables.

Variable definitions and coding schemes are checked against
the WVS Wave 7 questionnaire and codebook.

In [104]:
demographic_variables = (
    master_variable_dictionary[
        master_variable_dictionary["Category"]
        == "Demographic & Background"
    ][
        [
            "Variable",
            "Feature_group",
            "Variable_role",
            "Model_status",
            "Data type",
            "Unique values (excluding NaN)",
            "Candidate special-code details",
            "Missing percentage (NaN)"
        ]
    ]
    .sort_values(
        ["Feature_group", "Variable"]
    )
    .reset_index(drop=True)
)

print("Number of demographic variables:", len(demographic_variables))

demographic_variables

Number of demographic variables: 36


,Variable,Feature_group,Variable_role,Model_status,Data type,Unique values (excluding NaN),Candidate special-code details,Missing percentage (NaN)
0,Q260,Basic Demographic Background,Predictor candidate,Candidate,int64,5,-1: 3; -2: 39; -5: 53,0.0
1,Q261,Basic Demographic Background,Predictor candidate,Candidate,int64,92,-1: 9; -2: 445; -5: 1130,0.0
2,Q262,Basic Demographic Background,Predictor candidate,Candidate,int64,89,-1: 16; -2: 230; -4: 1; -5: 264,0.0
3,Q263,Basic Demographic Background,Predictor candidate,Candidate,int64,5,-1: 41; -2: 157; -5: 186,0.0
4,Q264,Basic Demographic Background,Predictor candidate,Candidate,int64,7,-1: 183; -2: 423; -3: 698; -4: 3036; -5: 341,0.0
5,Q265,Basic Demographic Background,Predictor candidate,Candidate,int64,7,-1: 261; -2: 458; -3: 746; -4: 3036; -5: 378,0.0
6,Q266,Basic Demographic Background,Predictor candidate,Candidate,int64,158,-1: 29; -2: 151; -4: 3036; -5: 175,0.0
7,Q267,Basic Demographic Background,Predictor candidate,Candidate,int64,163,-1: 83; -2: 969; -4: 6092; -5: 315,0.0
8,Q268,Basic Demographic Background,Predictor candidate,Candidate,int64,158,-1: 115; -2: 998; -4: 6092; -5: 378,0.0
9,Q269,Basic Demographic Background,Predictor candidate,Candidate,int64,6,-1: 37; -2: 140; -4: 4849; -5: 178,0.0


In [105]:
demographic_variable_names = demographic_variables["Variable"].tolist()

demographic_raw = survey_data[
    demographic_variable_names
].copy()

for variable in demographic_variable_names:
    print(f"\n{'=' * 60}")
    print(f"Variable: {variable}")
    print("Unique values:")
    print(
        sorted(
            demographic_raw[variable]
            .dropna()
            .unique()
            .tolist()
        )[:30]
    )


Variable: Q260
Unique values:
[-5, -2, -1, 1, 2]

Variable: Q261
Unique values:
[-5, -2, -1, 1916, 1917, 1919, 1921, 1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945]

Variable: Q262
Unique values:
[-5, -4, -2, -1, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]

Variable: Q263
Unique values:
[-5, -2, -1, 1, 2]

Variable: Q264
Unique values:
[-5, -4, -3, -2, -1, 1, 2]

Variable: Q265
Unique values:
[-5, -4, -3, -2, -1, 1, 2]

Variable: Q266
Unique values:
[-5, -4, -2, -1, 4, 8, 12, 16, 20, 24, 28, 31, 32, 36, 40, 48, 50, 51, 52, 56, 68, 70, 76, 100, 104, 112, 116, 120, 124, 144]

Variable: Q267
Unique values:
[-5, -4, -2, -1, 4, 8, 12, 16, 20, 24, 28, 31, 32, 36, 40, 44, 48, 50, 51, 52, 56, 68, 70, 76, 84, 92, 100, 104, 108, 112]

Variable: Q268
Unique values:
[-5, -4, -2, -1, 4, 8, 12, 16, 20, 24, 28, 31, 32, 36, 40, 44, 48, 50, 51, 52, 56, 6

In [106]:
demographic_measurement_mapping = {
    "Q260": "Nominal",
    "Q261": "Numeric",
    "Q262": "Numeric",
    "Q263": "Binary",
    "Q264": "Binary",
    "Q265": "Binary",
    "Q266": "Nominal",
    "Q267": "Nominal",
    "Q268": "Nominal",
    "Q269": "Nominal",
    "Q270": "Numeric",
    "Q271": "Nominal",
    "Q272": "Nominal",
    "Q273": "Nominal",
    "Q274": "Numeric",
    "Q275": "Ordinal",
    "Q275A": "Nominal",
    "Q275R": "Ordinal",
    "Q276": "Ordinal",
    "Q276A": "Nominal",
    "Q276R": "Ordinal",
    "Q277": "Ordinal",
    "Q277A": "Nominal",
    "Q277R": "Ordinal",
    "Q278": "Ordinal",
    "Q278A": "Nominal",
    "Q278R": "Ordinal",
    "Q279": "Nominal",
    "Q280": "Nominal",
    "Q281": "Nominal",
    "Q282": "Nominal",
    "Q283": "Nominal",
    "Q284": "Ordinal",
    "Q289": "Nominal",
    "Q289CS9": "Nominal",
    "Q290": "Nominal"
}

print(
    "Variables classified:",
    len(demographic_measurement_mapping)
)

Variables classified: 36


In [107]:
for variable, measurement_type in demographic_measurement_mapping.items():

    master_variable_dictionary.loc[
        master_variable_dictionary["Variable"] == variable,
        "Measurement_type"
    ] = measurement_type

In [108]:
demographic_check = (
    master_variable_dictionary[
        master_variable_dictionary["Category"]
        == "Demographic & Background"
    ][
        ["Variable", "Feature_group", "Measurement_type"]
    ]
)

demographic_check

,Variable,Feature_group,Measurement_type
290,Q260,Basic Demographic Background,Nominal
291,Q261,Basic Demographic Background,Numeric
292,Q262,Basic Demographic Background,Numeric
293,Q263,Basic Demographic Background,Binary
294,Q264,Basic Demographic Background,Binary
295,Q265,Basic Demographic Background,Binary
296,Q266,Basic Demographic Background,Nominal
297,Q267,Basic Demographic Background,Nominal
298,Q268,Basic Demographic Background,Nominal
299,Q269,Basic Demographic Background,Nominal


In [109]:
assert demographic_check["Measurement_type"].notna().all()

print("All demographic variables have measurement types.")

All demographic variables have measurement types.


In [110]:
measurement_coverage = (
    master_variable_dictionary
    .assign(
        Measurement_classified=lambda x:
            x["Measurement_type"].notna()
    )
    .groupby("Category")
    .agg(
        Total_variables=("Variable", "count"),
        Classified_variables=("Measurement_classified", "sum")
    )
    .reset_index()
)

measurement_coverage["Unclassified_variables"] = (
    measurement_coverage["Total_variables"]
    - measurement_coverage["Classified_variables"]
)

measurement_coverage["Coverage_percentage"] = (
    measurement_coverage["Classified_variables"]
    / measurement_coverage["Total_variables"]
    * 100
)

measurement_coverage.sort_values(
    "Coverage_percentage"
)

,Category,Total_variables,Classified_variables,Unclassified_variables,Coverage_percentage
3,Political & Institutional,154,0,154,0.000000
7,Target Variable,1,0,1,0.000000
5,Security,18,0,18,0.000000
4,Science & Technology,6,0,6,0.000000
2,Personal Values & Wellbeing,55,16,39,29.090909
6,Social & Community,84,32,52,38.095238
1,Economic,18,7,11,38.888889
0,Demographic & Background,36,36,0,100.000000


## 20. Economic Measurement Classification

This section examines economic security, economic values,
and household economic position variables.

In [111]:
economic_variables = (
    master_variable_dictionary[
        master_variable_dictionary["Category"] == "Economic"
    ][
        [
            "Variable",
            "Feature_group",
            "Variable_role",
            "Model_status",
            "Data type",
            "Unique values (excluding NaN)",
            "Candidate special-code details",
            "Missing percentage (NaN)"
        ]
    ]
    .sort_values(["Feature_group", "Variable"])
    .reset_index(drop=True)
)

print("Number of economic variables:", len(economic_variables))

economic_variables

Number of economic variables: 18


,Variable,Feature_group,Variable_role,Model_status,Data type,Unique values (excluding NaN),Candidate special-code details,Missing percentage (NaN)
0,Q50,Economic Security,Predictor candidate,Candidate,int64,13,-1: 287; -2: 230; -5: 108,0.0
1,Q51,Economic Security,Predictor candidate,Candidate,int64,7,-1: 184; -2: 170; -5: 162,0.0
2,Q52,Economic Security,Predictor candidate,Candidate,int64,7,-1: 247; -2: 195; -5: 145,0.0
3,Q53,Economic Security,Predictor candidate,Candidate,int64,7,-1: 194; -2: 210; -5: 217,0.0
4,Q54,Economic Security,Predictor candidate,Candidate,int64,7,-1: 236; -2: 238; -5: 145,0.0
5,Q55,Economic Security,Predictor candidate,Candidate,int64,7,-1: 287; -2: 254; -5: 205,0.0
6,Q56,Economic Security,Predictor candidate,Candidate,int64,6,-1: 1259; -2: 351; -5: 150,0.0
7,Q106,Economic Values,Predictor candidate,Candidate,int64,13,-1: 903; -2: 297; -5: 169,0.0
8,Q107,Economic Values,Predictor candidate,Candidate,int64,13,-1: 2753; -2: 497; -5: 194,0.0
9,Q108,Economic Values,Predictor candidate,Candidate,int64,13,-1: 717; -2: 292; -5: 169,0.0


In [112]:
economic_variable_names = economic_variables["Variable"].tolist()

for variable in economic_variable_names:
    print(f"\n{'=' * 50}")
    print(f"Variable: {variable}")
    print(
        sorted(
            survey_data[variable]
            .dropna()
            .unique()
            .tolist()
        )[:40]
    )


Variable: Q50
[-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Variable: Q51
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q52
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q53
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q54
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q55
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q56
[-5, -2, -1, 1, 2, 3]

Variable: Q106
[-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Variable: Q107
[-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Variable: Q108
[-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Variable: Q109
[-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Variable: Q110
[-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Variable: Q111
[-5, -2, -1, 1, 2, 3]

Variable: Q285
[-5, -2, -1, 1, 2]

Variable: Q286
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q287
[-5, -4, -2, -1, 1, 2, 3, 4, 5]

Variable: Q288
[-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Variable: Q288R
[-5, -2, -1, 1, 2, 3]


### 20.1 Economic Measurement Type Classification

Measurement types are assigned based on the response scales
and question definitions in the WVS Wave 7 questionnaire
and codebook.

In [113]:
economic_measurement_mapping = {
    "Q50": "Ordinal",
    "Q51": "Ordinal",
    "Q52": "Ordinal",
    "Q53": "Ordinal",
    "Q54": "Ordinal",
    "Q55": "Ordinal",
    "Q56": "Ordinal",

    "Q106": "Ordinal",
    "Q107": "Ordinal",
    "Q108": "Ordinal",
    "Q109": "Ordinal",
    "Q110": "Ordinal",
    "Q111": "Nominal",

    "Q285": "Binary",
    "Q286": "Ordinal",
    "Q287": "Ordinal",
    "Q288": "Ordinal",
    "Q288R": "Ordinal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Measurement_type"]
    .fillna(
        master_variable_dictionary["Variable"].map(
            economic_measurement_mapping
        )
    )
)

print(
    "Economic variables classified:",
    master_variable_dictionary[
        "Variable"
    ].isin(economic_measurement_mapping).sum()
)

Economic variables classified: 18


In [114]:
economic_measurement_check = (
    master_variable_dictionary[
        master_variable_dictionary["Variable"].isin(
            economic_measurement_mapping
        )
    ][
        ["Variable", "Feature_group", "Measurement_type"]
    ]
    .sort_values(["Feature_group", "Variable"])
    .reset_index(drop=True)
)

economic_measurement_check

,Variable,Feature_group,Measurement_type
0,Q50,Economic Security,Ordinal
1,Q51,Economic Security,Ordinal
2,Q52,Economic Security,Ordinal
3,Q53,Economic Security,Ordinal
4,Q54,Economic Security,Ordinal
5,Q55,Economic Security,Ordinal
6,Q56,Economic Security,Ordinal
7,Q106,Economic Values,Ordinal
8,Q107,Economic Values,Ordinal
9,Q108,Economic Values,Ordinal


In [115]:
social_unclassified = (
    master_variable_dictionary[
        (master_variable_dictionary["Category"] == "Social & Community")
        &
        (master_variable_dictionary["Measurement_type"].isna())
    ][
        [
            "Variable",
            "Feature_group",
            "Variable_role",
            "Model_status",
            "Data type",
            "Unique values (excluding NaN)",
            "Candidate special-code details",
            "Missing percentage (NaN)"
        ]
    ]
    .sort_values(["Feature_group", "Variable"])
    .reset_index(drop=True)
)

print("Unclassified Social & Community variables:",
      len(social_unclassified))

social_unclassified

Unclassified Social & Community variables: 52


,Variable,Feature_group,Variable_role,Model_status,Data type,Unique values (excluding NaN),Candidate special-code details,Missing percentage (NaN)
0,Q33_3,Family and Gender Norms,Alternative version,Review before modeling,int64,6,-1: 503; -2: 122; -5: 111,0.0
1,Q34_3,Family and Gender Norms,Alternative version,Review before modeling,int64,6,-1: 626; -2: 177; -5: 110,0.0
2,Q35_3,Family and Gender Norms,Alternative version,Review before modeling,int64,6,-1: 1140; -2: 199; -5: 125,0.0
3,Q57,General and interpersonal trust,Predictor candidate,Candidate,int64,5,-1: 737; -2: 310; -5: 290,0.0
4,Q58,General and interpersonal trust,Predictor candidate,Candidate,int64,7,-1: 102; -2: 125; -5: 105,0.0
5,Q59,General and interpersonal trust,Predictor candidate,Candidate,int64,7,-1: 534; -2: 184; -5: 109,0.0
6,Q60,General and interpersonal trust,Predictor candidate,Candidate,int64,7,-1: 284; -2: 195; -5: 106,0.0
7,Q61,General and interpersonal trust,Predictor candidate,Candidate,int64,7,-1: 1009; -2: 223; -5: 129,0.0
8,Q62,General and interpersonal trust,Predictor candidate,Candidate,int64,7,-1: 3049; -2: 705; -5: 322,0.0
9,Q63,General and interpersonal trust,Predictor candidate,Candidate,int64,7,-1: 3182; -2: 632; -5: 264,0.0


In [116]:
trust_variables = [
    "Q57", "Q58", "Q59", "Q60",
    "Q61", "Q62", "Q63"
]

for variable in trust_variables:
    print(f"\n{'=' * 50}")
    print(f"Variable: {variable}")

    print(
        sorted(
            survey_data[variable]
            .dropna()
            .unique()
            .tolist()
        )
    )
    


Variable: Q57
[-5, -2, -1, 1, 2]

Variable: Q58
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q59
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q60
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q61
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q62
[-5, -2, -1, 1, 2, 3, 4]

Variable: Q63
[-5, -2, -1, 1, 2, 3, 4]


In [117]:
trust_measurement_mapping = {
    "Q57": "Binary",
    "Q58": "Ordinal",
    "Q59": "Ordinal",
    "Q60": "Ordinal",
    "Q61": "Ordinal",
    "Q62": "Ordinal",
    "Q63": "Ordinal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Measurement_type"]
    .fillna(
        master_variable_dictionary["Variable"].map(
            trust_measurement_mapping
        )
    )
)

In [118]:
trust_measurement_check = (
    master_variable_dictionary[
        master_variable_dictionary["Variable"].isin(
            trust_measurement_mapping
        )
    ][["Variable", "Feature_group", "Measurement_type"]]
    .sort_values("Variable")
    .reset_index(drop=True)
)

trust_measurement_check

,Variable,Feature_group,Measurement_type
0,Q57,General and interpersonal trust,Binary
1,Q58,General and interpersonal trust,Ordinal
2,Q59,General and interpersonal trust,Ordinal
3,Q60,General and interpersonal trust,Ordinal
4,Q61,General and interpersonal trust,Ordinal
5,Q62,General and interpersonal trust,Ordinal
6,Q63,General and interpersonal trust,Ordinal


In [119]:
immigration_variables = [
    f"Q{i}" for i in range(121, 131)
]

for variable in immigration_variables:
    print(f"\n{'=' * 50}")
    print(f"Variable: {variable}")

    print(
        sorted(
            survey_data[variable]
            .dropna()
            .unique()
            .tolist()
        )
    )


Variable: Q121
[-5, -2, -1, 1, 2, 3, 4, 5]

Variable: Q122
[-5, -2, -1, 0, 1, 2]

Variable: Q123
[-5, -2, -1, 0, 1, 2]

Variable: Q124
[-5, -2, -1, 0, 1, 2]

Variable: Q125
[-5, -4, -2, -1, 0, 1, 2]

Variable: Q126
[-5, -2, -1, 0, 1, 2]

Variable: Q127
[-5, -2, -1, 0, 1, 2]

Variable: Q128
[-5, -2, -1, 0, 1, 2]

Variable: Q129
[-5, -2, -1, 0, 1, 2]

Variable: Q130
[-5, -4, -2, -1, 1, 2, 3, 4]


In [120]:
immigration_measurement_mapping = {
    "Q121": "Ordinal",
    "Q122": "Ordinal",
    "Q123": "Ordinal",
    "Q124": "Ordinal",
    "Q125": "Ordinal",
    "Q126": "Ordinal",
    "Q127": "Ordinal",
    "Q128": "Ordinal",
    "Q129": "Ordinal",
    "Q130": "Ordinal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Measurement_type"]
    .fillna(
        master_variable_dictionary["Variable"].map(
            immigration_measurement_mapping
        )
    )
)

In [121]:
immigration_measurement_check = (
    master_variable_dictionary[
        master_variable_dictionary["Variable"].isin(
            immigration_measurement_mapping
        )
    ][
        ["Variable", "Feature_group", "Measurement_type"]
    ]
    .sort_values("Variable")
    .reset_index(drop=True)
)

immigration_measurement_check

,Variable,Feature_group,Measurement_type
0,Q121,Immigration Attitudes,Ordinal
1,Q122,Immigration Attitudes,Ordinal
2,Q123,Immigration Attitudes,Ordinal
3,Q124,Immigration Attitudes,Ordinal
4,Q125,Immigration Attitudes,Ordinal
5,Q126,Immigration Attitudes,Ordinal
6,Q127,Immigration Attitudes,Ordinal
7,Q128,Immigration Attitudes,Ordinal
8,Q129,Immigration Attitudes,Ordinal
9,Q130,Immigration Attitudes,Ordinal


In [122]:
news_variables = [
    f"Q{i}" for i in range(201, 209)
]

for variable in news_variables:
    print(f"\n{'=' * 50}")
    print(f"Variable: {variable}")

    print(
        sorted(
            survey_data[variable]
            .dropna()
            .unique()
            .tolist()
        )
    )


Variable: Q201
[-5, -2, -1, 1, 2, 3, 4, 5]

Variable: Q202
[-5, -3, -2, -1, 1, 2, 3, 4, 5]

Variable: Q203
[-5, -2, -1, 1, 2, 3, 4, 5]

Variable: Q204
[-5, -2, -1, 1, 2, 3, 4, 5]

Variable: Q205
[-5, -2, -1, 1, 2, 3, 4, 5]

Variable: Q206
[-5, -2, -1, 1, 2, 3, 4, 5]

Variable: Q207
[-5, -4, -2, -1, 1, 2, 3, 4, 5]

Variable: Q208
[-5, -2, -1, 1, 2, 3, 4, 5]


In [123]:
news_measurement_mapping = {
    "Q201": "Ordinal",
    "Q202": "Ordinal",
    "Q203": "Ordinal",
    "Q204": "Ordinal",
    "Q205": "Ordinal",
    "Q206": "Ordinal",
    "Q207": "Ordinal",
    "Q208": "Ordinal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Measurement_type"]
    .fillna(
        master_variable_dictionary["Variable"].map(
            news_measurement_mapping
        )
    )
)

In [124]:
news_measurement_check = (
    master_variable_dictionary[
        master_variable_dictionary["Variable"].isin(
            news_measurement_mapping
        )
    ][
        ["Variable", "Feature_group", "Measurement_type"]
    ]
    .sort_values("Variable")
    .reset_index(drop=True)
)

news_measurement_check

,Variable,Feature_group,Measurement_type
0,Q201,News and Information Sources,Ordinal
1,Q202,News and Information Sources,Ordinal
2,Q203,News and Information Sources,Ordinal
3,Q204,News and Information Sources,Ordinal
4,Q205,News and Information Sources,Ordinal
5,Q206,News and Information Sources,Ordinal
6,Q207,News and Information Sources,Ordinal
7,Q208,News and Information Sources,Ordinal


In [125]:
membership_variables = [
    f"Q{i}" for i in range(94, 106)
]

for variable in membership_variables:
    print(f"\n{'=' * 50}")
    print(f"Variable: {variable}")

    print(
        sorted(
            survey_data[variable]
            .dropna()
            .unique()
            .tolist()
        )
    )


Variable: Q94
[-5, -2, -1, 0, 1, 2]

Variable: Q95
[-5, -2, -1, 0, 1, 2]

Variable: Q96
[-5, -2, -1, 0, 1, 2]

Variable: Q97
[-5, -2, -1, 0, 1, 2]

Variable: Q98
[-5, -2, -1, 0, 1, 2]

Variable: Q99
[-5, -2, -1, 0, 1, 2]

Variable: Q100
[-5, -2, -1, 0, 1, 2]

Variable: Q101
[-5, -2, -1, 0, 1, 2]

Variable: Q102
[-5, -2, -1, 0, 1, 2]

Variable: Q103
[-5, -2, -1, 0, 1, 2]

Variable: Q104
[-5, -2, -1, 0, 1, 2]

Variable: Q105
[-5, -2, -1, 0, 1, 2]


In [126]:
membership_measurement_mapping = {
    f"Q{i}": "Ordinal"
    for i in range(94, 106)
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Measurement_type"]
    .fillna(
        master_variable_dictionary["Variable"].map(
            membership_measurement_mapping
        )
    )
)

In [127]:
membership_measurement_check = (
    master_variable_dictionary[
        master_variable_dictionary["Variable"].isin(
            membership_measurement_mapping
        )
    ][
        ["Variable", "Feature_group", "Measurement_type"]
    ]
    .sort_values("Variable")
    .reset_index(drop=True)
)

membership_measurement_check

,Variable,Feature_group,Measurement_type
0,Q100,Organizational membership,Ordinal
1,Q101,Organizational membership,Ordinal
2,Q102,Organizational membership,Ordinal
3,Q103,Organizational membership,Ordinal
4,Q104,Organizational membership,Ordinal
5,Q105,Organizational membership,Ordinal
6,Q94,Organizational membership,Ordinal
7,Q95,Organizational membership,Ordinal
8,Q96,Organizational membership,Ordinal
9,Q97,Organizational membership,Ordinal


In [128]:
alternative_variables = [
    "Q33_3",
    "Q34_3",
    "Q35_3"
]

for variable in alternative_variables:
    print(f"\n{'=' * 50}")
    print(f"Variable: {variable}")

    print(
        sorted(
            survey_data[variable]
            .dropna()
            .unique()
            .tolist()
        )
    )


Variable: Q33_3
[-5, -2, -1, 1, 2, 3]

Variable: Q34_3
[-5, -2, -1, 1, 2, 3]

Variable: Q35_3
[-5, -2, -1, 1, 2, 3]


In [129]:
alternative_measurement_mapping = {
    "Q33_3": "Ordinal",
    "Q34_3": "Ordinal",
    "Q35_3": "Ordinal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Measurement_type"]
    .fillna(
        master_variable_dictionary["Variable"].map(
            alternative_measurement_mapping
        )
    )
)

alternative_measurement_check = (
    master_variable_dictionary[
        master_variable_dictionary["Variable"].isin(
            alternative_measurement_mapping
        )
    ][
        ["Variable", "Feature_group", "Variable_role", "Measurement_type"]
    ]
    .sort_values("Variable")
    .reset_index(drop=True)
)

alternative_measurement_check

,Variable,Feature_group,Variable_role,Measurement_type
0,Q33_3,Family and Gender Norms,Alternative version,Ordinal
1,Q34_3,Family and Gender Norms,Alternative version,Ordinal
2,Q35_3,Family and Gender Norms,Alternative version,Ordinal


In [130]:
recoded_membership_variables = [
    f"Q{i}R" for i in range(94, 106)
]

for variable in recoded_membership_variables:
    print(f"\n{'=' * 50}")
    print(f"Variable: {variable}")

    print(
        sorted(
            survey_data[variable]
            .dropna()
            .unique()
            .tolist()
        )
    )


Variable: Q94R
[-5, -2, -1, 0, 1]

Variable: Q95R
[-5, -2, -1, 0, 1]

Variable: Q96R
[-5, -2, -1, 0, 1]

Variable: Q97R
[-5, -2, -1, 0, 1]

Variable: Q98R
[-5, -2, -1, 0, 1]

Variable: Q99R
[-5, -2, -1, 0, 1]

Variable: Q100R
[-5, -2, -1, 0, 1]

Variable: Q101R
[-5, -2, -1, 0, 1]

Variable: Q102R
[-5, -2, -1, 0, 1]

Variable: Q103R
[-5, -2, -1, 0, 1]

Variable: Q104R
[-5, -2, -1, 0, 1]

Variable: Q105R
[-5, -2, -1, 0, 1]


In [131]:
recoded_membership_mapping = {
    f"Q{i}R": "Binary"
    for i in range(94, 106)
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Measurement_type"]
    .fillna(
        master_variable_dictionary["Variable"].map(
            recoded_membership_mapping
        )
    )
)

recoded_membership_check = (
    master_variable_dictionary[
        master_variable_dictionary["Variable"].isin(
            recoded_membership_mapping
        )
    ][
        ["Variable", "Feature_group", "Measurement_type"]
    ]
    .sort_values("Variable")
    .reset_index(drop=True)
)

recoded_membership_check

,Variable,Feature_group,Measurement_type
0,Q100R,Recoded membership variables,Binary
1,Q101R,Recoded membership variables,Binary
2,Q102R,Recoded membership variables,Binary
3,Q103R,Recoded membership variables,Binary
4,Q104R,Recoded membership variables,Binary
5,Q105R,Recoded membership variables,Binary
6,Q94R,Recoded membership variables,Binary
7,Q95R,Recoded membership variables,Binary
8,Q96R,Recoded membership variables,Binary
9,Q97R,Recoded membership variables,Binary


In [132]:
social_coverage = (
    master_variable_dictionary[
        master_variable_dictionary["Category"] == "Social & Community"
    ]
    .groupby("Feature_group")
    .agg(
        Total_variables=("Variable", "count"),
        Classified_variables=("Measurement_type", "count")
    )
)

social_coverage["Coverage_percentage"] = (
    social_coverage["Classified_variables"]
    / social_coverage["Total_variables"]
    * 100
).round(2)

social_coverage

,Total_variables,Classified_variables,Coverage_percentage
Feature_group,,,
Child-rearing Values,11,11,100.0
Family and Gender Norms,15,15,100.0
General and interpersonal trust,7,7,100.0
Immigration Attitudes,10,10,100.0
News and Information Sources,8,8,100.0
Organizational membership,12,12,100.0
Recoded membership variables,12,12,100.0
Social Tolerance,9,9,100.0


In [133]:
social_unclassified = master_variable_dictionary[
    (master_variable_dictionary["Category"] == "Social & Community")
    & (master_variable_dictionary["Measurement_type"].isna())
]

social_unclassified[
    ["Variable", "Feature_group", "Variable_role"]
]

,Variable,Feature_group,Variable_role


In [134]:
overall_measurement_coverage = (
    master_variable_dictionary
    .groupby("Category")
    .agg(
        Total_variables=("Variable", "count"),
        Classified_variables=("Measurement_type", "count")
    )
)

overall_measurement_coverage["Unclassified_variables"] = (
    overall_measurement_coverage["Total_variables"]
    - overall_measurement_coverage["Classified_variables"]
)

overall_measurement_coverage["Coverage_percentage"] = (
    overall_measurement_coverage["Classified_variables"]
    / overall_measurement_coverage["Total_variables"]
    * 100
).round(2)

overall_measurement_coverage

,Total_variables,Classified_variables,Unclassified_variables,Coverage_percentage
Category,,,,
Demographic & Background,36,36,0,100.00
Economic,18,18,0,100.00
Personal Values & Wellbeing,55,16,39,29.09
Political & Institutional,154,0,154,0.00
Science & Technology,6,0,6,0.00
Security,18,0,18,0.00
Social & Community,84,84,0,100.00
Target Variable,1,0,1,0.00


In [135]:
total_variables = len(master_variable_dictionary)

classified_variables = (
    master_variable_dictionary["Measurement_type"].notna().sum()
)

print(f"Total variables: {total_variables}")
print(f"Classified variables: {classified_variables}")
print(f"Unclassified variables: {total_variables - classified_variables}")
print(
    f"Overall coverage: "
    f"{classified_variables / total_variables * 100:.2f}%"
)

Total variables: 372
Classified variables: 154
Unclassified variables: 218
Overall coverage: 41.40%


In [136]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [137]:
personal_unclassified = master_variable_dictionary[
    (master_variable_dictionary["Category"] == "Personal Values & Wellbeing")
    & (master_variable_dictionary["Measurement_type"].isna())
]

personal_unclassified_summary = (
    personal_unclassified
    .groupby("Feature_group")
    .agg(
        Variable_count=("Variable", "count"),
        Variables=("Variable", lambda x: ", ".join(x))
    )
    .sort_values("Feature_group")
)

personal_unclassified_summary

,Variable_count,Variables
Feature_group,,
Ethical Values and Moral Norms,20,"Q176, Q177, Q178, Q179, Q180, Q181, Q182, Q183..."
Postmaterialist Priorities,6,"Q152, Q153, Q154, Q155, Q156, Q157"
Religious Beliefs,7,"Q164, Q165, Q166, Q167, Q168, Q169, Q170"
Religious Practices and Identity,6,"Q171, Q172, Q172R, Q173, Q174, Q175"


In [138]:
personal_groups = {
    "Ethical Values and Moral Norms": [
        f"Q{i}" for i in range(176, 196)
    ],
    "Postmaterialist Priorities": [
        f"Q{i}" for i in range(152, 158)
    ],
    "Religious Beliefs": [
        f"Q{i}" for i in range(164, 171)
    ],
    "Religious Practices and Identity": [
        "Q171", "Q172", "Q172R",
        "Q173", "Q174", "Q175"
    ]
}

for group, variables in personal_groups.items():
    print(f"\n{'#' * 60}")
    print(f"FEATURE GROUP: {group}")
    print(f"{'#' * 60}")

    for variable in variables:
        values = sorted(
            survey_data[variable]
            .dropna()
            .unique()
            .tolist()
        )

        print(f"{variable}: {values}")


############################################################
FEATURE GROUP: Ethical Values and Moral Norms
############################################################
Q176: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q177: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q178: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q179: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q180: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q181: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q182: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q183: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q184: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q185: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q186: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q187: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q188: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q189: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q190: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q191: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q192: [-5, 

In [139]:
personal_measurement_mapping = {}

# Ethical Values and Moral Norms
for i in range(176, 196):
    personal_measurement_mapping[f"Q{i}"] = "Ordinal"

# Postmaterialist Priorities
for i in range(152, 158):
    personal_measurement_mapping[f"Q{i}"] = "Nominal"

# Religious Beliefs
personal_measurement_mapping.update({
    "Q164": "Ordinal",
    "Q165": "Binary",
    "Q166": "Binary",
    "Q167": "Binary",
    "Q168": "Binary",
    "Q169": "Ordinal",
    "Q170": "Ordinal"
})

# Religious Practices and Identity
personal_measurement_mapping.update({
    "Q171": "Ordinal",
    "Q172": "Ordinal",
    "Q172R": "Ordinal",
    "Q173": "Nominal",
    "Q174": "Nominal",
    "Q175": "Nominal"
})

# Update master variable dictionary
master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Measurement_type"]
    .fillna(
        master_variable_dictionary["Variable"].map(
            personal_measurement_mapping
        )
    )
)

# Verify
personal_measurement_check = (
    master_variable_dictionary[
        master_variable_dictionary["Variable"].isin(
            personal_measurement_mapping
        )
    ][
        ["Variable", "Feature_group", "Measurement_type"]
    ]
    .sort_values("Variable")
    .reset_index(drop=True)
)

personal_measurement_check

,Variable,Feature_group,Measurement_type
0,Q152,Postmaterialist Priorities,Nominal
1,Q153,Postmaterialist Priorities,Nominal
2,Q154,Postmaterialist Priorities,Nominal
3,Q155,Postmaterialist Priorities,Nominal
4,Q156,Postmaterialist Priorities,Nominal
5,Q157,Postmaterialist Priorities,Nominal
6,Q164,Religious Beliefs,Ordinal
7,Q165,Religious Beliefs,Binary
8,Q166,Religious Beliefs,Binary
9,Q167,Religious Beliefs,Binary


In [140]:
personal_coverage = (
    master_variable_dictionary[
        master_variable_dictionary["Category"]
        == "Personal Values & Wellbeing"
    ]
    .groupby("Feature_group")
    .agg(
        Total_variables=("Variable", "count"),
        Classified_variables=("Measurement_type", "count")
    )
)

personal_coverage["Coverage_percentage"] = (
    personal_coverage["Classified_variables"]
    / personal_coverage["Total_variables"]
    * 100
).round(2)

personal_coverage

,Total_variables,Classified_variables,Coverage_percentage
Feature_group,,,
Ethical Values and Moral Norms,20,20,100.0
Personal Values,6,6,100.0
Postmaterialist Priorities,6,6,100.0
Religious Beliefs,7,7,100.0
Religious Practices and Identity,6,6,100.0
Wellbeing,3,3,100.0
Work and Social Attitudes,7,7,100.0


In [141]:
personal_unclassified = master_variable_dictionary[
    (master_variable_dictionary["Category"]
     == "Personal Values & Wellbeing")
    & (master_variable_dictionary["Measurement_type"].isna())
]

personal_unclassified[
    ["Variable", "Feature_group", "Variable_role"]
]

,Variable,Feature_group,Variable_role


In [142]:
political_unclassified = master_variable_dictionary[
    (master_variable_dictionary["Category"] == "Political & Institutional")
    & (master_variable_dictionary["Measurement_type"].isna())
]

political_unclassified_summary = (
    political_unclassified
    .groupby("Feature_group")
    .agg(
        Variable_count=("Variable", "count"),
        Variables=("Variable", lambda x: ", ".join(x))
    )
    .sort_values("Feature_group")
)

political_unclassified_summary

,Variable_count,Variables
Feature_group,,
Civic and Political Activism,4,"Q213, Q214, Q215, Q216"
Conventional Political Participation,4,"Q209, Q210, Q211, Q212"
Country-specific institutional confidence,17,"Q82_AFRICANUNION, Q82_APEC, Q82_ARABLEAGUE, Q8..."
Electoral Integrity and Political Voice,12,"Q224, Q225, Q226, Q227, Q228, Q229, Q230, Q231..."
"Freedom, Equality and Political Values",3,"Q149, Q150, Q151"
Government Surveillance Attitudes,3,"Q196, Q197, Q198"
Institutional Confidence,1,Q82
Institutional confidence,25,"Q64, Q65, Q66, Q67, Q68, Q69, Q70, Q71, Q72, Q..."
International organization attitudes,1,Q90


In [143]:
print(
    "Unclassified political variables:",
    len(political_unclassified)
)

Unclassified political variables: 154


In [144]:
participation_groups = {
    "Conventional Political Participation": [
        f"Q{i}" for i in range(209, 213)
    ],
    "Civic and Political Activism": [
        f"Q{i}" for i in range(213, 217)
    ],
    "Online Political Participation": [
        f"Q{i}" for i in range(217, 221)
    ]
}

for group, variables in participation_groups.items():
    print(f"\n{'#' * 55}")
    print(f"FEATURE GROUP: {group}")
    print(f"{'#' * 55}")

    for variable in variables:
        values = sorted(
            survey_data[variable]
            .dropna()
            .unique()
            .tolist()
        )

        print(f"{variable}: {values}")


#######################################################
FEATURE GROUP: Conventional Political Participation
#######################################################
Q209: [-5, -2, -1, 1, 2, 3]
Q210: [-5, -2, -1, 1, 2, 3]
Q211: [-5, -2, -1, 1, 2, 3]
Q212: [-5, -2, -1, 1, 2, 3]

#######################################################
FEATURE GROUP: Civic and Political Activism
#######################################################
Q213: [-5, -2, -1, 1, 2, 3]
Q214: [-5, -2, -1, 1, 2, 3]
Q215: [-5, -4, -2, -1, 1, 2, 3]
Q216: [-5, -2, -1, 1, 2, 3]

#######################################################
FEATURE GROUP: Online Political Participation
#######################################################
Q217: [-5, -4, -2, -1, 1, 2, 3]
Q218: [-5, -4, -2, -1, 1, 2, 3]
Q219: [-5, -4, -2, -1, 1, 2, 3]
Q220: [-5, -4, -2, -1, 1, 2, 3]


In [145]:
# Political participation and activism
participation_measurement_mapping = {}

for i in range(209, 221):
    participation_measurement_mapping[f"Q{i}"] = "Ordinal"

# Update master variable dictionary
master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(participation_measurement_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

# Verify
participation_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        participation_measurement_mapping
    )
][["Variable", "Feature_group", "Measurement_type"]]

participation_check

,Variable,Feature_group,Measurement_type
236,Q209,Conventional Political Participation,Ordinal
237,Q210,Conventional Political Participation,Ordinal
238,Q211,Conventional Political Participation,Ordinal
239,Q212,Conventional Political Participation,Ordinal
240,Q213,Civic and Political Activism,Ordinal
241,Q214,Civic and Political Activism,Ordinal
242,Q215,Civic and Political Activism,Ordinal
243,Q216,Civic and Political Activism,Ordinal
244,Q217,Online Political Participation,Ordinal
245,Q218,Online Political Participation,Ordinal


In [146]:
for variable in ["Q199", "Q200"]:
    values = sorted(
        survey_data[variable]
        .dropna()
        .unique()
        .tolist()
    )

    print(f"{variable}: {values}")

Q199: [-5, -2, -1, 1, 2, 3, 4]
Q200: [-5, -2, -1, 1, 2, 3]


In [147]:
political_interest_mapping = {
    "Q199": "Ordinal",
    "Q200": "Ordinal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(political_interest_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

# Verify
master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        political_interest_mapping
    )
][["Variable", "Feature_group", "Measurement_type"]]

,Variable,Feature_group,Measurement_type
226,Q199,Political Interest and Discussion,Ordinal
227,Q200,Political Interest and Discussion,Ordinal


In [148]:
institutional_confidence_variables = (
    [f"Q{i}" for i in range(64, 82)]
    + [f"Q{i}" for i in range(83, 90)]
    + ["Q82"]
)

for variable in institutional_confidence_variables:
    values = sorted(
        survey_data[variable]
        .dropna()
        .unique()
        .tolist()
    )

    print(f"{variable}: {values}")

Q64: [-5, -2, -1, 1, 2, 3, 4]
Q65: [-5, -4, -2, -1, 1, 2, 3, 4]
Q66: [-5, -2, -1, 1, 2, 3, 4]
Q67: [-5, -2, -1, 1, 2, 3, 4]
Q68: [-5, -2, -1, 1, 2, 3, 4]
Q69: [-5, -4, -2, -1, 1, 2, 3, 4]
Q70: [-5, -4, -2, -1, 1, 2, 3, 4]
Q71: [-5, -4, -2, -1, 1, 2, 3, 4]
Q72: [-5, -2, -1, 1, 2, 3, 4]
Q73: [-5, -2, -1, 1, 2, 3, 4]
Q74: [-5, -2, -1, 1, 2, 3, 4]
Q75: [-5, -2, -1, 1, 2, 3, 4]
Q76: [-5, -4, -2, -1, 1, 2, 3, 4]
Q77: [-5, -2, -1, 1, 2, 3, 4]
Q78: [-5, -2, -1, 1, 2, 3, 4]
Q79: [-5, -2, -1, 1, 2, 3, 4]
Q80: [-5, -2, -1, 1, 2, 3, 4]
Q81: [-5, -2, -1, 1, 2, 3, 4]
Q83: [-5, -4, -2, -1, 1, 2, 3, 4]
Q84: [-5, -4, -2, -1, 1, 2, 3, 4]
Q85: [-5, -4, -2, -1, 1, 2, 3, 4]
Q86: [-5, -4, -2, -1, 1, 2, 3, 4]
Q87: [-5, -4, -2, -1, 1, 2, 3, 4]
Q88: [-5, -4, -2, -1, 1, 2, 3, 4]
Q89: [-5, -4, -2, -1, 1, 2, 3, 4]
Q82: [-5, -4, -2, -1, 1, 2, 3, 4]


In [149]:
# Institutional Confidence
institutional_confidence_mapping = {}

institutional_confidence_variables = (
    [f"Q{i}" for i in range(64, 82)]
    + ["Q82"]
    + [f"Q{i}" for i in range(83, 90)]
)

for variable in institutional_confidence_variables:
    institutional_confidence_mapping[variable] = "Ordinal"

# Update master dictionary
master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(institutional_confidence_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

# Verify
institutional_confidence_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        institutional_confidence_variables
    )
][["Variable", "Feature_group", "Measurement_type"]]

institutional_confidence_check

,Variable,Feature_group,Measurement_type
62,Q64,Institutional confidence,Ordinal
63,Q65,Institutional confidence,Ordinal
64,Q66,Institutional confidence,Ordinal
65,Q67,Institutional confidence,Ordinal
66,Q68,Institutional confidence,Ordinal
67,Q69,Institutional confidence,Ordinal
68,Q70,Institutional confidence,Ordinal
69,Q71,Institutional confidence,Ordinal
70,Q72,Institutional confidence,Ordinal
71,Q73,Institutional confidence,Ordinal


In [150]:
for i in range(112, 121):
    variable = f"Q{i}"

    values = sorted(
        survey_data[variable]
        .dropna()
        .unique()
        .tolist()
    )

    print(f"{variable}: {values}")

Q112: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q113: [-5, -4, -2, -1, 1, 2, 3, 4]
Q114: [-5, -4, -2, -1, 1, 2, 3, 4]
Q115: [-5, -4, -2, -1, 1, 2, 3, 4]
Q116: [-5, -4, -2, -1, 1, 2, 3, 4]
Q117: [-5, -4, -2, -1, 1, 2, 3, 4]
Q118: [-5, -4, -2, -1, 1, 2, 3, 4]
Q119: [-5, -4, -2, -1, 0, 1, 2, 3, 4]
Q120: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [151]:
# Perceived Corruption
corruption_measurement_mapping = {
    "Q112": "Ordinal",
    "Q113": "Ordinal",
    "Q114": "Ordinal",
    "Q115": "Ordinal",
    "Q116": "Ordinal",
    "Q117": "Ordinal",
    "Q118": "Ordinal",
    "Q119": "Nominal",
    "Q120": "Ordinal"
}

# Update master dictionary
master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(corruption_measurement_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

# Verify
corruption_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        corruption_measurement_mapping
    )
][["Variable", "Feature_group", "Measurement_type"]]

corruption_check

,Variable,Feature_group,Measurement_type
138,Q112,Perceived Corruption,Ordinal
139,Q113,Perceived Corruption,Ordinal
140,Q114,Perceived Corruption,Ordinal
141,Q115,Perceived Corruption,Ordinal
142,Q116,Perceived Corruption,Ordinal
143,Q117,Perceived Corruption,Ordinal
144,Q118,Perceived Corruption,Ordinal
145,Q119,Perceived Corruption,Nominal
146,Q120,Perceived Corruption,Ordinal


In [152]:
for i in range(149, 152):
    variable = f"Q{i}"

    values = sorted(
        survey_data[variable]
        .dropna()
        .unique()
        .tolist()
    )

    print(f"{variable}: {values}")

Q149: [-5, -2, -1, 1, 2]
Q150: [-5, -2, -1, 1, 2]
Q151: [-5, -4, -2, -1, 1, 2]


In [3]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("../WVS_analysis_output")

master_variable_dictionary = pd.read_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv"
)

print(master_variable_dictionary.shape)

(372, 18)


In [4]:
# Freedom, Equality and Political Values
freedom_values_mapping = {
    "Q149": "Binary",
    "Q150": "Binary",
    "Q151": "Binary"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(freedom_values_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

# Verify
master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        freedom_values_mapping
    )
][["Variable", "Feature_group", "Measurement_type"]]

,Variable,Feature_group,Measurement_type
175,Q149,"Freedom, Equality and Political Values",Binary
176,Q150,"Freedom, Equality and Political Values",Binary
177,Q151,"Freedom, Equality and Political Values",Binary


In [5]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [8]:
import pandas as pd
from pathlib import Path

DATA_FILE = Path("../data/WVS_Cross-National_Wave_7_csv_v6_0.csv")

survey_data = pd.read_csv(
    DATA_FILE,
    usecols=["Q196", "Q197", "Q198"]
)

print(survey_data.shape)

(97220, 3)


In [9]:
for i in range(196, 199):
    variable = f"Q{i}"
    values = sorted(
        survey_data[variable].dropna().unique().tolist()
    )
    print(f"{variable}: {values}")

Q196: [-5, -4, -2, -1, 1, 2, 3, 4]
Q197: [-5, -4, -2, -1, 1, 2, 3, 4]
Q198: [-5, -4, -2, -1, 1, 2, 3, 4]


In [10]:
surveillance_values_mapping = {
    "Q196": "Ordinal",
    "Q197": "Ordinal",
    "Q198": "Ordinal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(surveillance_values_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        surveillance_values_mapping
    )
][["Variable", "Feature_group", "Measurement_type"]]

,Variable,Feature_group,Measurement_type
223,Q196,Government Surveillance Attitudes,Ordinal
224,Q197,Government Surveillance Attitudes,Ordinal
225,Q198,Government Surveillance Attitudes,Ordinal


In [11]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [12]:
security_variables = [
    f"Q{i}" for i in range(131, 149)
]

security_data = pd.read_csv(
    DATA_FILE,
    usecols=security_variables
)

print(security_data.shape)

(97220, 18)


In [13]:
for variable in security_variables:
    values = sorted(
        security_data[variable].dropna().unique().tolist()
    )
    print(f"{variable}: {values}")

Q131: [-5, -2, -1, 1, 2, 3, 4]
Q132: [-5, -4, -2, -1, 1, 2, 3, 4]
Q133: [-5, -4, -2, -1, 1, 2, 3, 4]
Q134: [-5, -4, -2, -1, 1, 2, 3, 4]
Q135: [-5, -4, -2, -1, 1, 2, 3, 4]
Q136: [-5, -4, -2, -1, 1, 2, 3, 4]
Q137: [-5, -4, -2, -1, 1, 2, 3, 4]
Q138: [-5, -4, -2, -1, 1, 2, 3, 4]
Q139: [-5, -4, -2, -1, 1, 2]
Q140: [-5, -4, -2, -1, 1, 2]
Q141: [-5, -4, -2, -1, 1, 2]
Q142: [-5, -2, -1, 1, 2, 3, 4]
Q143: [-5, -2, -1, 1, 2, 3, 4]
Q144: [-5, -4, -2, -1, 1, 2]
Q145: [-5, -4, -2, -1, 1, 2]
Q146: [-5, -4, -2, -1, 1, 2, 3, 4]
Q147: [-5, -4, -2, -1, 1, 2, 3, 4]
Q148: [-5, -4, -2, -1, 1, 2, 3, 4]


In [14]:
security_values_mapping = {}

# Q131
security_values_mapping["Q131"] = "Ordinal"

# Q132–Q138
for i in range(132, 139):
    security_values_mapping[f"Q{i}"] = "Ordinal"

# Q139–Q141
for i in range(139, 142):
    security_values_mapping[f"Q{i}"] = "Binary"

# Q142–Q143
for i in range(142, 144):
    security_values_mapping[f"Q{i}"] = "Ordinal"

# Q144–Q145
for i in range(144, 146):
    security_values_mapping[f"Q{i}"] = "Binary"

# Q146–Q148
for i in range(146, 149):
    security_values_mapping[f"Q{i}"] = "Ordinal"


master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(security_values_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [15]:
security_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        security_values_mapping
    )
]

print(security_check[
    ["Variable", "Feature_group", "Measurement_type"]
].to_string(index=False))

print("\nTotal classified:", security_check["Measurement_type"].notna().sum())

Variable                       Feature_group Measurement_type
    Q131  Personal and Neighborhood Security          Ordinal
    Q132  Personal and Neighborhood Security          Ordinal
    Q133  Personal and Neighborhood Security          Ordinal
    Q134  Personal and Neighborhood Security          Ordinal
    Q135  Personal and Neighborhood Security          Ordinal
    Q136  Personal and Neighborhood Security          Ordinal
    Q137  Personal and Neighborhood Security          Ordinal
    Q138  Personal and Neighborhood Security          Ordinal
    Q139  Personal and Neighborhood Security           Binary
    Q140  Personal and Neighborhood Security           Binary
    Q141  Personal and Neighborhood Security           Binary
    Q142 Security Concerns and Victimization          Ordinal
    Q143 Security Concerns and Victimization          Ordinal
    Q144 Security Concerns and Victimization           Binary
    Q145 Security Concerns and Victimization           Binary
    Q146

In [16]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [17]:
science_variables = [
    f"Q{i}" for i in range(158, 164)
]

science_data = pd.read_csv(
    DATA_FILE,
    usecols=science_variables
)

print(science_data.shape)

(97220, 6)


In [18]:
for variable in science_variables:
    values = sorted(
        science_data[variable].dropna().unique().tolist()
    )
    print(f"{variable}: {values}")

Q158: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q159: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q160: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q161: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q162: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q163: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [19]:
science_values_mapping = {
    "Q158": "Ordinal",
    "Q159": "Ordinal",
    "Q160": "Ordinal",
    "Q161": "Ordinal",
    "Q162": "Ordinal",
    "Q163": "Ordinal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(science_values_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [20]:
science_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        science_values_mapping
    )
]

print(science_check[
    ["Variable", "Feature_group", "Measurement_type"]
].to_string(index=False))

print("\nTotal classified:",
      science_check["Measurement_type"].notna().sum())

Variable                    Feature_group Measurement_type
    Q158 Science and Technology Attitudes          Ordinal
    Q159 Science and Technology Attitudes          Ordinal
    Q160 Science and Technology Attitudes          Ordinal
    Q161 Science and Technology Attitudes          Ordinal
    Q162 Science and Technology Attitudes          Ordinal
    Q163 Science and Technology Attitudes          Ordinal

Total classified: 6


In [21]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [22]:
political_variables = master_variable_dictionary[
    master_variable_dictionary["Category"] == "Political & Institutional"
].copy()

political_unclassified = political_variables[
    political_variables["Measurement_type"].isna()
]

print("Total political variables:",
      len(political_variables))

print("Already classified:",
      political_variables["Measurement_type"].notna().sum())

print("Not yet classified:",
      len(political_unclassified))

Total political variables: 154
Already classified: 6
Not yet classified: 148


In [23]:
political_unclassified_summary = (
    political_unclassified
    .groupby("Feature_group")["Variable"]
    .agg(["count", list])
    .reset_index()
)

print(political_unclassified_summary.to_string(index=False))

                                  Feature_group  count                                                                                                                                                                                             list
                   Civic and Political Activism      4                                                                                                                                                                         [Q213, Q214, Q215, Q216]
           Conventional Political Participation      4                                                                                                                                                                         [Q209, Q210, Q211, Q212]
      Country-specific institutional confidence     17 [Q82_AFRICANUNION, Q82_APEC, Q82_ARABLEAGUE, Q82_ASEAN, Q82_CIS, Q82_CUSMA, Q82_ECO, Q82_EU, Q82_GULFCOOP, Q82_ISLCOOP, Q82_MERCOSUR, Q82_NAFTA, Q82_OAS, Q82_SAARC, Q82_SCO, Q82_TLC, Q82_UNDP]
        

In [24]:
institutional_variables = (
    [f"Q{i}" for i in range(64, 82)]
    + ["Q82"]
    + [f"Q{i}" for i in range(83, 90)]
)

institutional_data = pd.read_csv(
    DATA_FILE,
    usecols=institutional_variables
)

print(institutional_data.shape)

(97220, 26)


In [25]:
for variable in institutional_variables:
    values = sorted(
        institutional_data[variable]
        .dropna()
        .unique()
        .tolist()
    )
    print(f"{variable}: {values}")

Q64: [-5, -2, -1, 1, 2, 3, 4]
Q65: [-5, -4, -2, -1, 1, 2, 3, 4]
Q66: [-5, -2, -1, 1, 2, 3, 4]
Q67: [-5, -2, -1, 1, 2, 3, 4]
Q68: [-5, -2, -1, 1, 2, 3, 4]
Q69: [-5, -4, -2, -1, 1, 2, 3, 4]
Q70: [-5, -4, -2, -1, 1, 2, 3, 4]
Q71: [-5, -4, -2, -1, 1, 2, 3, 4]
Q72: [-5, -2, -1, 1, 2, 3, 4]
Q73: [-5, -2, -1, 1, 2, 3, 4]
Q74: [-5, -2, -1, 1, 2, 3, 4]
Q75: [-5, -2, -1, 1, 2, 3, 4]
Q76: [-5, -4, -2, -1, 1, 2, 3, 4]
Q77: [-5, -2, -1, 1, 2, 3, 4]
Q78: [-5, -2, -1, 1, 2, 3, 4]
Q79: [-5, -2, -1, 1, 2, 3, 4]
Q80: [-5, -2, -1, 1, 2, 3, 4]
Q81: [-5, -2, -1, 1, 2, 3, 4]
Q82: [-5, -4, -2, -1, 1, 2, 3, 4]
Q83: [-5, -4, -2, -1, 1, 2, 3, 4]
Q84: [-5, -4, -2, -1, 1, 2, 3, 4]
Q85: [-5, -4, -2, -1, 1, 2, 3, 4]
Q86: [-5, -4, -2, -1, 1, 2, 3, 4]
Q87: [-5, -4, -2, -1, 1, 2, 3, 4]
Q88: [-5, -4, -2, -1, 1, 2, 3, 4]
Q89: [-5, -4, -2, -1, 1, 2, 3, 4]


In [26]:
institutional_values_mapping = {
    variable: "Ordinal"
    for variable in institutional_variables
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(institutional_values_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [27]:
institutional_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        institutional_variables
    )
]

print(institutional_check[
    ["Variable", "Feature_group", "Measurement_type"]
].to_string(index=False))

print(
    "\nTotal classified:",
    institutional_check["Measurement_type"].notna().sum()
)

Variable            Feature_group Measurement_type
     Q64 Institutional confidence          Ordinal
     Q65 Institutional confidence          Ordinal
     Q66 Institutional confidence          Ordinal
     Q67 Institutional confidence          Ordinal
     Q68 Institutional confidence          Ordinal
     Q69 Institutional confidence          Ordinal
     Q70 Institutional confidence          Ordinal
     Q71 Institutional confidence          Ordinal
     Q72 Institutional confidence          Ordinal
     Q73 Institutional confidence          Ordinal
     Q74 Institutional confidence          Ordinal
     Q75 Institutional confidence          Ordinal
     Q76 Institutional confidence          Ordinal
     Q77 Institutional confidence          Ordinal
     Q78 Institutional confidence          Ordinal
     Q79 Institutional confidence          Ordinal
     Q80 Institutional confidence          Ordinal
     Q81 Institutional confidence          Ordinal
     Q83 Institutional confiden

In [28]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [29]:
international_variables = ["Q90", "Q91", "Q92", "Q93"]

international_data = pd.read_csv(
    DATA_FILE,
    usecols=international_variables
)

for variable in international_variables:
    values = sorted(
        international_data[variable]
        .dropna()
        .unique()
        .tolist()
    )
    print(f"{variable}: {values}")

Q90: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q91: [-5, -4, -2, -1, 1, 2, 3]
Q92: [-5, -4, -2, -1, 1, 2, 3]
Q93: [-5, -4, -2, -1, 1, 2, 3]


In [30]:
international_values_mapping = {
    "Q90": "Ordinal",
    "Q91": "Nominal",
    "Q92": "Nominal",
    "Q93": "Nominal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(international_values_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [31]:
international_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        international_variables
    )
]

print(
    international_check[
        ["Variable", "Feature_group", "Measurement_type"]
    ].to_string(index=False)
)

Variable                        Feature_group Measurement_type
     Q90 International organization attitudes          Ordinal
     Q91 International organization knowledge          Nominal
     Q92 International organization knowledge          Nominal
     Q93 International organization knowledge          Nominal


In [32]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [33]:
corruption_variables = [
    f"Q{i}" for i in range(112, 121)
]

corruption_data = pd.read_csv(
    DATA_FILE,
    usecols=corruption_variables
)

for variable in corruption_variables:
    values = sorted(
        corruption_data[variable]
        .dropna()
        .unique()
        .tolist()
    )
    print(f"{variable}: {values}")

Q112: [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q113: [-5, -4, -2, -1, 1, 2, 3, 4]
Q114: [-5, -4, -2, -1, 1, 2, 3, 4]
Q115: [-5, -4, -2, -1, 1, 2, 3, 4]
Q116: [-5, -4, -2, -1, 1, 2, 3, 4]
Q117: [-5, -4, -2, -1, 1, 2, 3, 4]
Q118: [-5, -4, -2, -1, 1, 2, 3, 4]
Q119: [-5, -4, -2, -1, 0, 1, 2, 3, 4]
Q120: [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [34]:
corruption_values_mapping = {
    "Q112": "Ordinal",
    "Q113": "Ordinal",
    "Q114": "Ordinal",
    "Q115": "Ordinal",
    "Q116": "Ordinal",
    "Q117": "Ordinal",
    "Q118": "Ordinal",
    "Q119": "Nominal",
    "Q120": "Ordinal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(corruption_values_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [35]:
corruption_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        corruption_variables
    )
]

print(
    corruption_check[
        ["Variable", "Feature_group", "Measurement_type"]
    ].to_string(index=False)
)

Variable        Feature_group Measurement_type
    Q112 Perceived Corruption          Ordinal
    Q113 Perceived Corruption          Ordinal
    Q114 Perceived Corruption          Ordinal
    Q115 Perceived Corruption          Ordinal
    Q116 Perceived Corruption          Ordinal
    Q117 Perceived Corruption          Ordinal
    Q118 Perceived Corruption          Ordinal
    Q119 Perceived Corruption          Nominal
    Q120 Perceived Corruption          Ordinal


In [36]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [37]:
political_interest_variables = ["Q199", "Q200"]

political_interest_data = pd.read_csv(
    DATA_FILE,
    usecols=political_interest_variables
)

for variable in political_interest_variables:
    values = sorted(
        political_interest_data[variable]
        .dropna()
        .unique()
        .tolist()
    )
    print(f"{variable}: {values}")

Q199: [-5, -2, -1, 1, 2, 3, 4]
Q200: [-5, -2, -1, 1, 2, 3]


In [38]:
political_interest_mapping = {
    "Q199": "Ordinal",
    "Q200": "Ordinal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(political_interest_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [39]:
political_interest_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        political_interest_variables
    )
]

print(
    political_interest_check[
        ["Variable", "Feature_group", "Measurement_type"]
    ].to_string(index=False)
)

Variable                     Feature_group Measurement_type
    Q199 Political Interest and Discussion          Ordinal
    Q200 Political Interest and Discussion          Ordinal


In [40]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [41]:
participation_variables = [
    f"Q{i}" for i in range(209, 221)
]

participation_data = pd.read_csv(
    DATA_FILE,
    usecols=participation_variables
)

for variable in participation_variables:
    values = sorted(
        participation_data[variable]
        .dropna()
        .unique()
        .tolist()
    )
    print(f"{variable}: {values}")

Q209: [-5, -2, -1, 1, 2, 3]
Q210: [-5, -2, -1, 1, 2, 3]
Q211: [-5, -2, -1, 1, 2, 3]
Q212: [-5, -2, -1, 1, 2, 3]
Q213: [-5, -2, -1, 1, 2, 3]
Q214: [-5, -2, -1, 1, 2, 3]
Q215: [-5, -4, -2, -1, 1, 2, 3]
Q216: [-5, -2, -1, 1, 2, 3]
Q217: [-5, -4, -2, -1, 1, 2, 3]
Q218: [-5, -4, -2, -1, 1, 2, 3]
Q219: [-5, -4, -2, -1, 1, 2, 3]
Q220: [-5, -4, -2, -1, 1, 2, 3]


In [42]:
participation_mapping = {
    variable: "Ordinal"
    for variable in participation_variables
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(participation_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [43]:
participation_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        participation_variables
    )
]

print(
    participation_check[
        ["Variable", "Feature_group", "Measurement_type"]
    ].to_string(index=False)
)

print(
    "\nTotal classified:",
    participation_check["Measurement_type"].notna().sum()
)

Variable                        Feature_group Measurement_type
    Q209 Conventional Political Participation          Ordinal
    Q210 Conventional Political Participation          Ordinal
    Q211 Conventional Political Participation          Ordinal
    Q212 Conventional Political Participation          Ordinal
    Q213         Civic and Political Activism          Ordinal
    Q214         Civic and Political Activism          Ordinal
    Q215         Civic and Political Activism          Ordinal
    Q216         Civic and Political Activism          Ordinal
    Q217       Online Political Participation          Ordinal
    Q218       Online Political Participation          Ordinal
    Q219       Online Political Participation          Ordinal
    Q220       Online Political Participation          Ordinal

Total classified: 12


In [44]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [45]:
party_mapping = {
    "Q221": "Nominal",
    "Q222": "Nominal",
    "Q223": "Nominal",
    "Q223_ABREV": "Nominal",
    "Q223_LOCAL": "Nominal"
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(party_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [46]:
party_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(party_mapping)
]

print(party_check[
    ["Variable", "Feature_group", "Measurement_type"]
].to_string(index=False))

master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

  Variable                                Feature_group Measurement_type
      Q221 Political Participation and Party Preference          Nominal
      Q222 Political Participation and Party Preference          Nominal
      Q223 Political Participation and Party Preference          Nominal
Q223_ABREV Political Participation and Party Preference          Nominal
Q223_LOCAL Political Participation and Party Preference          Nominal


In [47]:
electoral_variables = (
    [f"Q{i}" for i in range(224, 235)]
    + ["Q234A"]
)

electoral_mapping = {
    variable: "Ordinal"
    for variable in electoral_variables
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(electoral_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [48]:
electoral_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        electoral_variables
    )
]

print(
    electoral_check[
        ["Variable", "Feature_group", "Measurement_type"]
    ].to_string(index=False)
)

print(
    "\nTotal classified:",
    electoral_check["Measurement_type"].notna().sum()
)

Variable                           Feature_group Measurement_type
    Q224 Electoral Integrity and Political Voice          Ordinal
    Q225 Electoral Integrity and Political Voice          Ordinal
    Q226 Electoral Integrity and Political Voice          Ordinal
    Q227 Electoral Integrity and Political Voice          Ordinal
    Q228 Electoral Integrity and Political Voice          Ordinal
    Q229 Electoral Integrity and Political Voice          Ordinal
    Q230 Electoral Integrity and Political Voice          Ordinal
    Q231 Electoral Integrity and Political Voice          Ordinal
    Q232 Electoral Integrity and Political Voice          Ordinal
    Q233 Electoral Integrity and Political Voice          Ordinal
    Q234 Electoral Integrity and Political Voice          Ordinal
   Q234A Electoral Integrity and Political Voice          Ordinal

Total classified: 12


In [49]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [50]:
confirmed_ordinal_variables = (
    [f"Q{i}" for i in range(235, 250)]
    + [f"Q{i}" for i in range(255, 260)]
)

political_culture_mapping = {
    variable: "Ordinal"
    for variable in confirmed_ordinal_variables
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(political_culture_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [51]:
culture_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        confirmed_ordinal_variables
    )
]

print(culture_check[
    ["Variable", "Feature_group", "Measurement_type"]
].to_string(index=False))

print("\nTotal classified:", len(culture_check))

master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

Variable                                Feature_group Measurement_type
    Q235 Political Culture and Democratic Preferences          Ordinal
    Q236 Political Culture and Democratic Preferences          Ordinal
    Q237 Political Culture and Democratic Preferences          Ordinal
    Q238 Political Culture and Democratic Preferences          Ordinal
    Q239 Political Culture and Democratic Preferences          Ordinal
    Q240 Political Culture and Democratic Preferences          Ordinal
    Q241 Political Culture and Democratic Preferences          Ordinal
    Q242 Political Culture and Democratic Preferences          Ordinal
    Q243 Political Culture and Democratic Preferences          Ordinal
    Q244 Political Culture and Democratic Preferences          Ordinal
    Q245 Political Culture and Democratic Preferences          Ordinal
    Q246 Political Culture and Democratic Preferences          Ordinal
    Q247 Political Culture and Democratic Preferences          Ordinal
    Q2

In [52]:
uncertain_variables = [
    "Q250", "Q251", "Q252", "Q253", "Q254"
]

uncertain_data = pd.read_csv(
    DATA_FILE,
    usecols=uncertain_variables
)

for variable in uncertain_variables:
    print(
        variable,
        sorted(
            uncertain_data[variable]
            .dropna()
            .unique()
            .tolist()
        )
    )

Q250 [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q251 [-5, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q252 [-5, -4, -2, -1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Q253 [-5, -2, -1, 1, 2, 3, 4]
Q254 [-5, -2, -1, 1, 2, 3, 4, 5]


In [53]:
remaining_culture_variables = [
    "Q250", "Q251", "Q252", "Q253", "Q254"
]

remaining_culture_mapping = {
    variable: "Ordinal"
    for variable in remaining_culture_variables
}

master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(remaining_culture_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

In [54]:
culture_variables = [
    f"Q{i}" for i in range(235, 260)
]

culture_check = master_variable_dictionary[
    master_variable_dictionary["Variable"].isin(
        culture_variables
    )
]

print(
    culture_check[
        ["Variable", "Feature_group", "Measurement_type"]
    ].to_string(index=False)
)

print("\nTotal classified:", len(culture_check))
print(
    "Unclassified:",
    culture_check["Measurement_type"].isna().sum()
)

Variable                                Feature_group Measurement_type
    Q235 Political Culture and Democratic Preferences          Ordinal
    Q236 Political Culture and Democratic Preferences          Ordinal
    Q237 Political Culture and Democratic Preferences          Ordinal
    Q238 Political Culture and Democratic Preferences          Ordinal
    Q239 Political Culture and Democratic Preferences          Ordinal
    Q240 Political Culture and Democratic Preferences          Ordinal
    Q241 Political Culture and Democratic Preferences          Ordinal
    Q242 Political Culture and Democratic Preferences          Ordinal
    Q243 Political Culture and Democratic Preferences          Ordinal
    Q244 Political Culture and Democratic Preferences          Ordinal
    Q245 Political Culture and Democratic Preferences          Ordinal
    Q246 Political Culture and Democratic Preferences          Ordinal
    Q247 Political Culture and Democratic Preferences          Ordinal
    Q2

In [55]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [56]:
print("Total variables:", len(master_variable_dictionary))

print("\nMeasurement Type distribution:")
print(
    master_variable_dictionary["Measurement_type"]
    .value_counts(dropna=False)
)

print(
    "\nUnclassified variables:",
    master_variable_dictionary["Measurement_type"].isna().sum()
)

Total variables: 372

Measurement Type distribution:
Measurement_type
Ordinal    199
NaN         93
Binary      45
Nominal     31
Numeric      4
Name: count, dtype: int64

Unclassified variables: 93


In [57]:
unclassified = master_variable_dictionary[
    master_variable_dictionary["Measurement_type"].isna()
].copy()

print("Unclassified variables:", len(unclassified))

print("\nUnclassified variables by Feature Group:")
print(
    unclassified["Feature_group"]
    .value_counts()
    .to_string()
)

print("\nVariable list by Feature Group:")

for group, data in unclassified.groupby("Feature_group"):
    print(f"\n{group} ({len(data)} variables)")
    print(", ".join(data["Variable"].tolist()))

Unclassified variables: 93

Unclassified variables by Feature Group:
Feature_group
Ethical Values and Moral Norms                     20
Political Trust - Parliament, Government and UN    18
Country-specific institutional confidence          17
Political Trust and Government Attitudes           15
Religious Beliefs                                   7
Postmaterialist Priorities                          6
Religious Practices and Identity                    6
Trust in Political Leaders                          3
Life Satisfaction                                   1

Variable list by Feature Group:

Country-specific institutional confidence (17 variables)
Q82_AFRICANUNION, Q82_APEC, Q82_ARABLEAGUE, Q82_ASEAN, Q82_CIS, Q82_CUSMA, Q82_ECO, Q82_EU, Q82_GULFCOOP, Q82_ISLCOOP, Q82_MERCOSUR, Q82_NAFTA, Q82_OAS, Q82_SAARC, Q82_SCO, Q82_TLC, Q82_UNDP

Ethical Values and Moral Norms (20 variables)
Q176, Q177, Q178, Q179, Q180, Q181, Q182, Q183, Q184, Q185, Q186, Q187, Q188, Q189, Q190, Q191, Q192, 

In [58]:
# 1. Country-specific institutional confidence
country_confidence = [
    col for col in master_variable_dictionary["Variable"]
    if col.startswith("Q82_")
]

# 2. Ethical Values and Moral Norms
ethical_values = [f"Q{i}" for i in range(176, 196)]

# 3. Political Trust - Parliament, Government and UN
political_trust = (
    [f"Q291P{i}" for i in range(1, 7)]
    + [f"Q291G{i}" for i in range(1, 7)]
    + [f"Q291UN{i}" for i in range(1, 7)]
)

# 4. Political Trust and Government Attitudes
political_attitudes = [
    f"Q292{letter}" for letter in "ABCDEFGHIJKLMNO"
]

# 5. Postmaterialist Priorities
postmaterialist = [f"Q{i}" for i in range(152, 158)]

# 6. Religious Beliefs
religious_beliefs = {
    "Q164": "Ordinal",
    "Q165": "Binary",
    "Q166": "Binary",
    "Q167": "Binary",
    "Q168": "Binary",
    "Q169": "Ordinal",
    "Q170": "Ordinal"
}

# 7. Religious Practices and Identity
religious_practices = {
    "Q171": "Ordinal",
    "Q172": "Ordinal",
    "Q172R": "Ordinal",
    "Q173": "Nominal",
    "Q174": "Nominal",
    "Q175": "Nominal"
}

# 8. Trust in Political Leaders
political_leaders = {
    "Q293": "Ordinal",
    "Q294A": "Numeric",
    "Q294B": "Numeric"
}

# 9. Measurement Type mapping
measurement_mapping = {}

for variable in country_confidence:
    measurement_mapping[variable] = "Ordinal"

for variable in ethical_values:
    measurement_mapping[variable] = "Ordinal"

for variable in political_trust:
    measurement_mapping[variable] = "Ordinal"

for variable in political_attitudes:
    measurement_mapping[variable] = "Ordinal"

for variable in postmaterialist:
    measurement_mapping[variable] = "Nominal"

measurement_mapping.update(religious_beliefs)
measurement_mapping.update(religious_practices)
measurement_mapping.update(political_leaders)

# Q49: Target variable
measurement_mapping["Q49"] = "Ordinal"

# Apply mapping
master_variable_dictionary["Measurement_type"] = (
    master_variable_dictionary["Variable"]
    .map(measurement_mapping)
    .fillna(master_variable_dictionary["Measurement_type"])
)

print("Variables mapped:", len(measurement_mapping))

Variables mapped: 93


In [59]:
print("Total variables:", len(master_variable_dictionary))

print("\nMeasurement Type distribution:")
print(
    master_variable_dictionary["Measurement_type"]
    .value_counts(dropna=False)
)

print(
    "\nUnclassified variables:",
    master_variable_dictionary["Measurement_type"].isna().sum()
)

Total variables: 372

Measurement Type distribution:
Measurement_type
Ordinal    277
Binary      49
Nominal     40
Numeric      6
Name: count, dtype: int64

Unclassified variables: 0


In [60]:
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

In [61]:
# Save the completed master variable dictionary
master_variable_dictionary.to_csv(
    OUTPUT_DIR / "WVS_Master_Variable_Dictionary.csv",
    index=False
)

print("Master Variable Dictionary saved successfully.")

Master Variable Dictionary saved successfully.
